## 1. Dataset Preparation

Load raw instance files and build the preprocessed data dictionary used by all solvers.


In [2]:
# Standard scientific and utility imports
import numpy as np
import os
from scipy.spatial.distance import cdist   # vectorised pairwise Euclidean distance
import re
import time
import gurobipy as gp
from gurobipy import GRB
import pulp                                # PuLP wrapper around CBC (open-source MIP)
from pathlib import Path
import tempfile
import csv
import random
import math
from copy import deepcopy


In [3]:
def load_instance(filepath):
    """
    Parse a CDSP benchmark instance file into a structured dictionary.

    File format
    -----------
    Line 0 : <char> <num_nodes> <max_vehicles>
        - num_nodes   : total node count including the depot (node 0)
        - max_vehicles: fleet size upper bound
    Lines 1+: <x> <y> <ready_time> <due_date>
        - Node 0 is the depot; nodes 1..num_nodes-1 are clinics (requests).
        - ready_time : earliest moment the vehicle may serve this node
        - due_date   : latest moment the vehicle must arrive at this node
        - max_tau is fixed at 480 (minutes) — the maximum full vehicle shift duration

    Returns
    -------
    data_dict with keys: coords, ready_times, due_dates,
                         max_vehicles, max_tau, num_nodes
    """
    with open(filepath, 'r') as f:
        lines = f.readlines()

    header = lines[0].strip().split()
    num_nodes    = int(header[1])
    max_vehicles = int(header[2])
    max_tau = 480          # fixed shift-duration limit (minutes)

    node_data = np.loadtxt(lines[1:])

    data_dict = {
        'coords'      : node_data[:, 0:2],
        'ready_times' : node_data[:, 2],
        'due_dates'   : node_data[:, 3],
        'max_vehicles': max_vehicles,
        'max_tau'     : max_tau,
        'num_nodes'   : num_nodes
    }
    return data_dict



In [4]:
def preprocess_instance(data_dict):
    """
    Build the arc-feasibility-filtered distance matrix and attach it to data_dict.

    Arc pruning logic
    -----------------
    An arc i→j is *infeasible* (set to inf) when a vehicle departing i at the
    earliest possible moment (ready_time[i]) cannot reach j before its due date:

        ready_time[i] + dist(i, j) > due_date[j]

    Removing these arcs up-front shrinks the MIP variable count and speeds up
    both the greedy heuristic and the ALNS insertion enumeration — infeasible
    arcs are simply never considered.

    Self-loops (i == i) are also forbidden (diagonal set to inf).

    Modifies data_dict in-place and returns it.
    """
    coords      = data_dict['coords']
    ready_times = data_dict['ready_times']
    due_dates   = data_dict['due_dates']

    # Full Euclidean distance matrix  (N × N)
    dist_matrix = cdist(coords, coords, metric='euclidean')

    # Earliest arrival at j if departing i exactly at ready_time[i]
    # Shape: (N, N) via broadcasting
    arrival_at_j = ready_times[:, np.newaxis] + dist_matrix

    # Mark arcs where arrival exceeds j's due date
    invalid_mask = arrival_at_j > due_dates[np.newaxis, :]

    # Self-loops are never valid
    np.fill_diagonal(invalid_mask, True)

    processed_dist_matrix = dist_matrix.copy()
    processed_dist_matrix[invalid_mask] = np.inf

    data_dict['dist_matrix'] = processed_dist_matrix
    return data_dict


## 2. Greedy Heuristic

Nearest-neighbour greedy construction with time-window awareness.

Used as a **standalone quick solution** and as a **warm-start seed** for both
the MIP solver and ALNS.  Returns `(routes, total_corrected_obj, unvisited)`.

Each route is `(list_of_clinic_indices, depot_return_time)`.
Clinics in `unvisited` could not be feasibly served by any vehicle; they receive
a penalty contribution so the objective stays comparable across instances.


In [5]:
def greedy_nearest_neighbor(data_dict):
    """
    Build a feasible solution by repeatedly extending each vehicle's route with
    the nearest clinic that can still be reached within its time window AND
    allows the vehicle to return to depot within the shift limit (max_tau).

    Key feasibility checks per candidate j
    ---------------------------------------
    1. Arc i→j must be finite (pruned by preprocess_instance).
    2. Arrival at j <= due_date[j].
    3. arrival[j] + dist(j, depot) <= depart_time + max_tau
       (the vehicle's total shift must not exceed max_tau minutes from departure).

    Departure time tracking
    -----------------------
    `depart_time` is computed once per route from the first clinic served:
        depart_time = max(0, ready_time[first_j] - dist(depot, first_j))
    This is the earliest the vehicle can leave the depot to arrive exactly at
    ready_time[first_j], and is used to enforce the shift-duration constraint.

    Unvisited clinics
    -----------------
    If a clinic cannot be added to any vehicle, it is left in `unvisited` and
    penalised in the objective as:
        due_date[j] + dist(j, depot) - ready_time[j]
    (an upper-bound proxy for what its completion time would cost).
    """
    dist = data_dict['dist_matrix']
    rt   = data_dict['ready_times']
    dd   = data_dict['due_dates']
    K    = data_dict['max_vehicles']
    T    = data_dict['max_tau']
    N    = data_dict['num_nodes']
    depot = 0

    unvisited = set(range(1, N))
    routes, total_C = [], 0.0

    for _ in range(K):
        if not unvisited:
            break
        route, current, curr_time = [], depot, 0.0
        depart_time = None

        while unvisited:
            best_j, best_arr = None, np.inf
            for j in unvisited:
                if np.isinf(dist[current, j]):
                    continue
                arr = max(curr_time + dist[current, j], rt[j])
                # Use known departure time for shift check; estimate it for the first node
                dep = depart_time if depart_time is not None else max(0.0, rt[j] - dist[depot, j])
                if arr <= dd[j] and arr + dist[j, depot] <= dep + T and arr < best_arr:
                    best_j, best_arr = j, arr
            if best_j is None:
                break
            if depart_time is None:
                depart_time = max(0.0, best_arr - dist[depot, best_j])
            route.append(best_j)
            unvisited.discard(best_j)
            curr_time, current = best_arr, best_j

        if route:
            return_time = curr_time + dist[current, depot]
            # Corrected completion-time contribution: Σ(C[j] - r[j]) for this route
            # C[j] = return_time for every j in the same leg (all share the depot return)
            total_C += return_time * len(route) - sum(rt[j] for j in route)
            routes.append((route, return_time))

    # Penalty for clinics that no vehicle could serve
    if unvisited:
        penalty = sum(
            dd[j] + (dist[j, depot] if not np.isinf(dist[j, depot]) else 0.0) - rt[j]
            for j in unvisited
        )
        total_C += penalty

    return routes, total_C, list(unvisited)


## 3. Exact MIP Model — strict Rocha et al. formulation

The Gurobi implementation in this notebook follows the two-index replenishment-arc formulation used by Rocha et al. (2025), rather than the earlier leg-based implementation.

### 3.1 Network representation

Let \(V=P\cup\{0\}\), where \(P\) is the set of pickup requests and node \(0\) is the laboratory/depot.

Three arc sets are used:

- \(A_0\): original arcs incident to the depot;
- \(A_P\): original request-to-request arcs;
- \(A_R\): replenishment arcs. A replenishment arc \(i\rightarrow j\) represents the physical movement \(i\rightarrow0\rightarrow j\), so a vehicle may return to the laboratory, unload specimens, and then start another trip without introducing an explicit trip index.

### 3.2 Variables

- \(x_e\): binary variable indicating whether arc \(e\) is used;
- \(y_{ij}\): binary specimen-tracking variable indicating whether specimen \(j\) is still carried after visiting request \(i\);
- \(z_j\): absolute visit time at request \(j\);
- \(\tau_j\): elapsed vehicle-shift time at request \(j\), measured continuously from the vehicle's first departure;
- \(C_j\): completion time of specimen \(j\), i.e. arrival time at the laboratory.

The reported benchmark objective is

\[
F'=\sum_{j\in P}(C_j-r_j),
\]

which differs from \(\sum_j C_j\) only by the instance-specific constant \(\sum_j r_j\). It therefore has the same minimizers.

### 3.3 Strict constraints implemented

The Gurobi model implements the Rocha formulation in the following order:

1. **Depot balance and fleet availability**: the number of depot departures equals the number of final returns and does not exceed \(K\).
2. **Request flow**: every request has exactly one incoming and one outgoing arc in the multigraph.
3. **Visit-time propagation** on both original and replenishment arcs.
4. **Hard time windows** \(r_j\le z_j\le d_j\).
5. **Specimen tracking** with \(y_{jj}=1\) and propagation only over original request-to-request arcs. Tracking stops at a replenishment arc because that arc contains a depot visit.
6. **Completion time**: if specimen \(j\) is still on the vehicle at request \(i\), then \(C_j\) cannot precede \(z_i+c_{i0}\).
7. **Shift-time initialization and propagation** through \(\tau_j\). The shift clock does **not** reset after a replenishment trip.
8. **Full-shift duration**: \(\tau_j+c_{j0}\le \tau^{max}\), so the complete sequence of trips assigned to one vehicle must fit within the shift limit.

### 3.4 Tightening and preprocessing

Before building the exact model, the request windows are tightened as

\[
r_j\leftarrow\max(r_j,c_{0j}),\qquad
d_j\leftarrow\min(d_j,d_0-c_{j0}).
\]

The model uses the paper-specific coefficients

\[
M_e=\max\{0,d_{s(e)}+c_e-r_{t(e)}\},
\]

\[
M_i=d_i+c_{i0},
\]

and

\[
M'_e=d_{t(e)}-c_{0,t(e)}.
\]

The exact solver rebuilds the Euclidean distance matrix from coordinates so that ALNS-specific arc pruning does not alter the benchmark MIP formulation.

### 3.5 Solver settings

- Gurobi is the primary exact solver.
- Warm starts are disabled for the strict replication because the anchor benchmark does not rely on a custom MIP start.
- The default full-instance time limit is configured in Section 5.
- CBC is retained only as a legacy fallback and should not be used for benchmark replication.


In [6]:
def parse_cbc_gap(log_text):
    """
    Extract the best objective value, lower bound, and MIP gap from CBC's log output.

    CBC does not expose these values through PuLP's API, so we parse the text log.
    Two log formats are attempted in order of reliability:

    1. Termination summary lines (most reliable, appear at the end of the log):
           'Objective value: <val>'
           'Lower bound: <val>'

    2. Search progress lines (fallback, emitted during the solve):
           '... best solution <val> best possible <val> ...'

    Returns
    -------
    (obj, lower_bound, gap)  — any value may be None if not found in the log.
    gap = |obj - lb| / |obj|  (relative MIP gap).
    """
    obj, lb = None, None

    # Pass 1: termination summary
    for line in log_text.splitlines():
        m = re.search(r'Objective value:\s*([\d.]+)', line)
        if m:
            obj = float(m.group(1))
        m = re.search(r'Lower bound:\s*([\d.]+)', line)
        if m:
            lb = float(m.group(1))

    # Pass 2: in-progress lines (if summary not found)
    if lb is None:
        for line in log_text.splitlines():
            m = re.search(
                r'best solution\s+([\d.]+)\s+best possible\s+([\d.]+)', line)
            if m:
                obj = float(m.group(1))
                lb  = float(m.group(2))

    gap = None
    if obj is not None and lb is not None and obj > 1e-9:
        gap = abs(obj - lb) / abs(obj)
    return obj, lb, gap

In [7]:
# ── Solver selection ──────────────────────────────────────────────────────────
# Change this flag to switch between solvers for all subsequent calls:
#   'GUROBI' → Gurobi Optimizer (requires a valid license; recommended for n > 25)
#   'CBC'    → open-source CBC via PuLP (no license required; slower on large instances)
MIP_SOLVER = 'GUROBI'   # ← change here


def mip_build_and_solve(data_dict, time_limit=3600, solver_name=None,
                        threads=4, verbose=True, use_warm_start=False):
    """
    Build and solve the CDSP MIP, dispatching to the selected backend.

    Parameters
    ----------
    data_dict      : preprocessed instance dict (from preprocess_instance)
    time_limit     : wall-clock seconds before the solver is interrupted
    solver_name    : 'GUROBI' or 'CBC'; None → use the global MIP_SOLVER flag
    threads        : solver thread count
    verbose        : whether to print solver log and progress
    use_warm_start : seed the MIP with a greedy solution to tighten the initial bound

    Returns
    -------
    dict with keys: status, objective, lower_bound, mip_gap_pct,
                    wall_time, proven_optimal, legs, warm_start_obj
    """
    if solver_name is None:
        solver_name = MIP_SOLVER
    if solver_name == 'GUROBI':
        return _mip_solve_gurobi(data_dict, time_limit=time_limit, threads=threads,
                                  verbose=verbose, use_warm_start=use_warm_start)
    else:
        return _mip_solve_cbc(data_dict, time_limit=time_limit, threads=threads,
                               verbose=verbose, use_warm_start=use_warm_start)


def _mip_solve_gurobi(data_dict, time_limit=3600, threads=4,
                      verbose=True, use_warm_start=False):
    """
    Exact two-index replenishment-arc formulation of Rocha et al. (2025).

    This implementation mirrors constraints (1)-(14) of the anchor paper.
    In particular it uses:
      x_e      unified binary arc variable on original and replenishment arcs
      y[i,j]   specimen-tracking binary variable
      z[j]     absolute visit time at request j
      tau[j]   elapsed time since the vehicle's first departure from the depot
      C[j]     request completion time (arrival at the laboratory)

    The benchmark-reported objective is F' = sum_j (C[j] - r[j]).  This differs
    from the mathematical objective sum C[j] only by an instance-specific
    constant, so it has exactly the same minimizers and LP relaxation strength.

    The paper-specific preprocessing is applied locally here:
        r_j <- max(r_j, c_0j)
        d_j <- min(d_j, d_0 - c_j0)
    This is intentionally kept separate from preprocess_instance(), which is
    also used by the ALNS and performs additional arc pruning.

    Notes on warm starts
    --------------------
    The anchor paper reports standard Gurobi parameters and does not describe a
    MIP start.  To keep the replication clean, warm starts are disabled by
    default.  If use_warm_start=True is passed, it is currently ignored rather
    than injecting a partial/possibly non-equivalent start.
    """
    coords = np.asarray(data_dict['coords'], dtype=float)
    rt0    = np.asarray(data_dict['ready_times'], dtype=float)
    dd0    = np.asarray(data_dict['due_dates'], dtype=float)
    K      = int(data_dict['max_vehicles'])
    T      = float(data_dict['max_tau'])
    N      = int(data_dict['num_nodes'])

    depot    = 0
    requests = list(range(1, N))

    # Rocha's graph is complete on the original arcs before time-window bounds
    # are enforced.  Rebuild Euclidean distances from coordinates so that the
    # ALNS-specific inf arc pruning in data_dict['dist_matrix'] does not change
    # the exact formulation.
    raw_dist = cdist(coords, coords, metric='euclidean')

    # Paper preprocessing: tighten request windows with depot reachability.
    r = rt0.copy()
    d = dd0.copy()
    d0 = float(dd0[depot])
    for j in requests:
        r[j] = max(r[j], raw_dist[depot, j])
        d[j] = min(d[j], d0 - raw_dist[j, depot])
        if r[j] > d[j] + 1e-9:
            return {
                'status': 'Infeasible', 'objective': None, 'lower_bound': None,
                'mip_gap_pct': None, 'wall_time': 0.0,
                'proven_optimal': False, 'formulation': 'Rocha2025'
            }

    # Arc sets.  The type tag is required because AP and AR are parallel arcs
    # between the same ordered pair of requests.
    A0 = ([('O', depot, j) for j in requests] +
          [('O', j, depot) for j in requests])
    AP = [('O', i, j) for i in requests for j in requests if i != j]
    AR = [('R', i, j) for i in requests for j in requests if i != j]
    A  = A0 + AP + AR

    def src(e): return e[1]
    def tgt(e): return e[2]

    c = {}
    for e in A0 + AP:
        c[e] = float(raw_dist[src(e), tgt(e)])
    for e in AR:
        i, j = src(e), tgt(e)
        c[e] = float(raw_dist[i, depot] + raw_dist[depot, j])

    out_arcs = {j: [e for e in A if src(e) == j] for j in range(N)}
    in_arcs  = {j: [e for e in A if tgt(e) == j] for j in range(N)}

    # Big-M coefficients exactly as defined in Rocha et al. (2025).
    M_e = {
        e: max(0.0, float(d[src(e)] + c[e] - r[tgt(e)]))
        for e in list(AP) + list(AR)
    }
    M_i = {
        i: float(d[i] + raw_dist[i, depot])
        for i in requests
    }
    Mprime_e = {
        e: float(d[tgt(e)] - raw_dist[depot, tgt(e)])
        for e in list(AP) + list(AR)
    }

    if use_warm_start and verbose:
        print('  Note: use_warm_start=True ignored for Rocha2025 replication.')

    env = gp.Env(empty=True)
    env.setParam('OutputFlag', 1 if verbose else 0)
    env.start()
    m = gp.Model(env=env)
    m.setParam('TimeLimit', time_limit)
    m.setParam('Threads', threads)

    # (12)-(14) variable domains.
    x   = m.addVars(A, vtype=GRB.BINARY, name='x')
    y   = m.addVars(requests, requests, vtype=GRB.BINARY, name='y')
    z   = m.addVars(requests, lb=0.0, vtype=GRB.CONTINUOUS, name='z')
    tau = m.addVars(requests, lb=0.0, vtype=GRB.CONTINUOUS, name='tau')
    C   = m.addVars(requests, lb=0.0, vtype=GRB.CONTINUOUS, name='C')

    # (1) Same minimizers as sum C_j, reported on Rocha's corrected F' scale.
    m.setObjective(gp.quicksum(C[j] - r[j] for j in requests), GRB.MINIMIZE)

    # (2) Depot flow balance and fleet limit.
    m.addConstr(
        gp.quicksum(x[e] for e in out_arcs[depot]) ==
        gp.quicksum(x[e] for e in in_arcs[depot]),
        name='c2_depot_balance')
    m.addConstr(
        gp.quicksum(x[e] for e in out_arcs[depot]) <= K,
        name='c2_fleet_limit')

    # (3) Every request has exactly one incoming and one outgoing arc in the
    # multigraph (original or replenishment).
    for j in requests:
        m.addConstr(gp.quicksum(x[e] for e in out_arcs[j]) == 1,
                    name=f'c3_out_{j}')
        m.addConstr(gp.quicksum(x[e] for e in in_arcs[j]) == 1,
                    name=f'c3_in_{j}')

    # (4) Visit-time propagation over original request-request arcs and
    # replenishment arcs.  Write every constraint as lhs <= 0 to avoid
    # NumPy/Gurobi comparison ambiguity.
    for e in list(AP) + list(AR):
        i = src(e)
        j = tgt(e)
        lhs = (
            z[i]
            + float(c[e])
            - z[j]
            - float(M_e[e]) * (1 - x[e])
        )
        m.addConstr(lhs <= 0, name=f'c4_{e[0]}_{i}_{j}')

    # (5) Tightened hard time windows.
    for j in requests:
        m.addConstr(z[j] >= float(r[j]), name=f'c5_r_{j}')
        m.addConstr(z[j] <= float(d[j]), name=f'c5_d_{j}')

    # (6) A request is on the vehicle immediately after collection.
    for j in requests:
        m.addConstr(y[j, j] == 1, name=f'c6_{j}')

    # (7) Propagate specimen j along ORIGINAL request-request arcs until the
    # vehicle next returns to the depot.  Replenishment arcs deliberately do
    # not propagate y because they contain a depot visit.
    for e in list(AP):
        i = src(e)
        k = tgt(e)
        for req in requests:
            if req == k:
                continue
            lhs = y[i, req] - y[k, req] - (1 - x[e])
            m.addConstr(lhs <= 0, name=f'c7_{i}_{k}_{req}')

    # (8) Completion time: if specimen req is still on the vehicle at i, its
    # delivery cannot occur before travelling i -> depot.
    for i in requests:
        return_to_depot = float(raw_dist[i, depot])
        Mi = float(M_i[i])
        for req in requests:
            lhs = (
                z[i]
                + return_to_depot
                - C[req]
                - Mi * (1 - y[i, req])
            )
            m.addConstr(lhs <= 0, name=f'c8_{i}_{req}')

    # (9) Relative visit time cannot be smaller than a direct depot -> j trip.
    # Put the Gurobi variable on the left-hand side; this avoids NumPy scalar
    # comparison creating an ambiguous TempConstr boolean.
    for j in requests:
        m.addConstr(
            tau[j] >= float(raw_dist[depot, j]),
            name=f'c9_{j}'
        )

    # (10) Propagate elapsed shift time over both AP and AR.  For a
    # replenishment arc, z[j]-z[i] includes the intermediate depot visit and
    # any waiting, so the shift clock does not reset between trips.
    for e in list(AP) + list(AR):
        i = src(e)
        j = tgt(e)
        lhs = (
            tau[i]
            + z[j]
            - z[i]
            - tau[j]
            - float(Mprime_e[e]) * (1 - x[e])
        )
        m.addConstr(lhs <= 0, name=f'c10_{e[0]}_{i}_{j}')

    # (11) Full vehicle shift from first departure to final depot return.
    for j in requests:
        m.addConstr(
            tau[j] + float(raw_dist[j, depot]) <= float(T),
            name=f'c11_{j}'
        )

    t0 = time.time()
    m.optimize()
    wall_time = time.time() - t0

    status_map = {
        GRB.OPTIMAL    : 'Optimal',
        GRB.INFEASIBLE : 'Infeasible',
        GRB.INF_OR_UNBD: 'Infeasible',
        GRB.TIME_LIMIT : 'Time limit',
        GRB.SUBOPTIMAL : 'Feasible',
    }
    status = status_map.get(m.Status, f'Status_{m.Status}')

    # Report Obj/LB on Rocha's original-release F' scale.
    reporting_shift = float(sum(r[j] - rt0[j] for j in requests))

    obj_val = (
        m.ObjVal + reporting_shift
        if m.SolCount > 0 else None
    )
    lower_bound = (
        None if m.Status == GRB.INFEASIBLE
        else m.ObjBound + reporting_shift
    )

    mip_gap = None
    if obj_val is not None and lower_bound is not None and abs(obj_val) > 1e-9:
        mip_gap = abs(obj_val - lower_bound) / abs(obj_val)

    truly_optimal = (m.Status == GRB.OPTIMAL and
                     (mip_gap is None or mip_gap < 1e-4))

    mip_legs = None
    mip_groups = None
    if m.SolCount > 0:
        mip_groups = _extract_groups_from_rocha_gurobi(x, requests, depot)
        mip_legs = [leg for grp in mip_groups for leg in grp]

    result = {
        'status'         : status,
        'objective'      : round(obj_val, 4) if obj_val is not None else None,
        'lower_bound'    : round(lower_bound, 4) if lower_bound is not None else None,
        'mip_gap_pct'    : round(mip_gap * 100, 2) if mip_gap is not None else None,
        'wall_time'      : round(wall_time, 2),
        'proven_optimal' : truly_optimal,
        'formulation'    : 'Rocha2025',
        'n_vars'         : m.NumVars,
        'n_constrs'      : m.NumConstrs,
    }
    if mip_legs is not None:
        result['legs'] = mip_legs
    if mip_groups is not None:
        result['route_groups'] = mip_groups

    m.dispose()
    env.dispose()
    return result



def _extract_groups_from_rocha_gurobi(x, requests, depot=0):
    """
    Reconstruct complete vehicle groups from Rocha's unified arc solution.

    Returns
    -------
    List[List[List[int]]]
        Outer list = vehicles.
        Middle list = depot-to-depot legs/trips of one vehicle.
        Inner list = request indices in one leg.

    A replenishment arc ('R', i, j) represents i -> depot -> j. It therefore
    closes the current leg but keeps the next leg on the SAME vehicle.
    """
    selected = {e for e in x.keys() if x[e].X > 0.5}
    starts = [j for j in requests if ('O', depot, j) in selected]

    route_groups = []
    globally_seen = set()

    for start in starts:
        cur = start
        vehicle_legs = []
        leg = []
        safety = 0

        while safety < 20 * (len(requests) + 1):
            safety += 1

            if cur in globally_seen:
                break

            globally_seen.add(cur)
            leg.append(cur)

            # Original request -> depot arc finishes this vehicle.
            if ('O', cur, depot) in selected:
                vehicle_legs.append(leg)
                leg = []
                break

            # Replenishment arc: finish this trip, continue same vehicle.
            next_repl = [
                j for j in requests
                if j != cur and ('R', cur, j) in selected
            ]

            # Original request -> request arc: remain on same trip.
            next_original = [
                j for j in requests
                if j != cur and ('O', cur, j) in selected
            ]

            if next_repl:
                vehicle_legs.append(leg)
                leg = []
                cur = next_repl[0]
                continue

            if next_original:
                cur = next_original[0]
                continue

            # Defensive fallback for an unexpected extraction state.
            if leg:
                vehicle_legs.append(leg)
            break

        if vehicle_legs:
            route_groups.append(vehicle_legs)

    return route_groups

def _extract_routes_from_rocha_gurobi(x, requests, depot):
    """Extract depot-to-depot legs from the unified Rocha arc variables."""
    selected = {e for e in x.keys() if x[e].X > 0.5}

    # Every route starts on an original depot->request arc.
    starts = [j for j in requests if ('O', depot, j) in selected]
    legs = []
    seen = set()

    for start in starts:
        cur = start
        leg = []
        safety = 0
        while cur != depot and safety < 10 * (len(requests) + 1):
            safety += 1
            # A request appears only once globally in a feasible solution.
            if cur in seen:
                break
            seen.add(cur)
            leg.append(cur)

            e_back = ('O', cur, depot)
            if e_back in selected:
                legs.append(leg)
                break

            next_original = [j for j in requests
                             if j != cur and ('O', cur, j) in selected]
            next_repl = [j for j in requests
                         if j != cur and ('R', cur, j) in selected]

            if next_repl:
                # The replenishment arc cur -> depot -> j closes the current
                # leg and immediately begins the next trip of the same vehicle.
                legs.append(leg)
                leg = []
                cur = next_repl[0]
            elif next_original:
                cur = next_original[0]
            else:
                # Defensive fallback for an unexpected extraction state.
                if leg:
                    legs.append(leg)
                break

    return legs

def _mip_solve_cbc(data_dict, time_limit=3600, threads=4,
                   verbose=True, use_warm_start=True):
    """
    CBC/PuLP fallback solver.  Identical formulation to _mip_solve_gurobi;
    differences are API-level only (PuLP vs gurobipy syntax).

    Notable differences vs. Gurobi backend
    ----------------------------------------
    - Warm-start hints use PuLP's setInitialValue() instead of .Start
    - The solver log is written to a temp file and parsed by parse_cbc_gap()
      because PuLP does not expose CBC's bound/gap programmatically
    - proven_optimal relies on both PuLP status string and parsed gap < 1e-4
    """
    dist = data_dict['dist_matrix']
    rt   = data_dict['ready_times']
    dd   = data_dict['due_dates']
    K    = data_dict['max_vehicles']
    T    = data_dict['max_tau']
    N    = data_dict['num_nodes']

    nodes    = list(range(N))
    requests = list(range(1, N))
    depot    = 0

    direct_arcs = [(i, j) for i in nodes for j in nodes
               if i != j and not np.isinf(dist[i, j])]
    replenishment_arcs = [(i, j) for i in requests for j in requests
               if i != j
               and not np.isinf(dist[i, depot])
               and not np.isinf(dist[depot, j])]

    M_direct        = {(i,j): max(dd[i] + dist[i,j] - rt[j], 0.0)
                       for (i,j) in direct_arcs}
    M_replenishment = {(i,j): max(dd[i] + dist[i,depot] + dist[depot,j] - rt[j], 0.0)
                       for (i,j) in replenishment_arcs}
    M_tau           = {i: max(dd[i] + dist[i, depot] - T, 0.0)
                       for i in requests if not np.isinf(dist[i, depot])}
    M_alpha         = {j: dd[j] + (dist[depot, j] if not np.isinf(dist[depot, j]) else 0.0)
                       for j in requests}
    M_C = {}
    for j in requests:
        feas_returns = [dist[j, depot]] if not np.isinf(dist[j, depot]) else []
        feas_returns += [dist[j,k] + dist[k, depot]
                         for k in requests if k != j
                         and not np.isinf(dist[j,k])
                         and not np.isinf(dist[k, depot])]
        M_C[j] = dd[j] + (max(feas_returns) if feas_returns else 0.0)
    M_C_global = max(M_C.values()) if M_C else 0.0

    prob  = pulp.LpProblem('CDSP_Vehicle', pulp.LpMinimize)
    x     = pulp.LpVariable.dicts('x',     direct_arcs,        cat='Binary')
    y     = pulp.LpVariable.dicts('y',     replenishment_arcs, cat='Binary')
    z     = pulp.LpVariable.dicts('z',     nodes,    lowBound=0, cat='Continuous')
    C     = pulp.LpVariable.dicts('C',     requests, lowBound=0, cat='Continuous')
    alpha = pulp.LpVariable.dicts('alpha', requests, lowBound=0, cat='Continuous')

    prob += pulp.lpSum(C[j] - rt[j] for j in requests), 'Total_corrected_completion_time'

    for j in requests:
        prob += (pulp.lpSum(x[i,j] for i in nodes    if (i,j) in x) +
                 pulp.lpSum(y[i,j] for i in requests if (i,j) in y) == 1, f'visit_{j}')
    for i in requests:
        prob += (pulp.lpSum(x[i,j] for j in nodes    if (i,j) in x) +
                 pulp.lpSum(y[i,j] for j in requests if (i,j) in y) == 1, f'leave_{i}')
    prob += (pulp.lpSum(x[depot,j] for j in requests if (depot,j) in x) <= K, 'max_vehicles')
    prob += (pulp.lpSum(x[i,depot] for i in requests if (i,depot) in x) ==
             pulp.lpSum(x[depot,j] for j in requests if (depot,j) in x), 'route_balance')

    prob += z[depot] == 0, 'depot_time'
    for (i,j) in direct_arcs:
        if j != depot:
            prob += (z[j] >= z[i] + dist[i,j] - M_direct[(i,j)] * (1 - x[i,j]),
                     f'time_direct_{i}_{j}')
    for (i,j) in replenishment_arcs:
        replenish = dist[i,depot] + dist[depot,j]
        prob += (z[j] >= z[i] + replenish - M_replenishment[(i,j)] * (1 - y[i,j]),
                 f'time_replenishment_{i}_{j}')
    for j in requests:
        prob += z[j] >= rt[j], f'tw_early_{j}'
        prob += z[j] <= dd[j], f'tw_late_{j}'

    for j in requests:
        if (depot, j) in x:
            prob += (alpha[j] >= z[j] - dist[depot,j] - M_alpha[j]*(1 - x[depot,j]), f'alpha_lb_{j}')
            prob += (alpha[j] <= z[j] - dist[depot,j] + M_alpha[j]*(1 - x[depot,j]), f'alpha_ub_{j}')
    for (i, j) in direct_arcs:
        if i != depot and j != depot:
            prob += (alpha[j] >= alpha[i] - M_alpha[j]*(1 - x[i,j]), f'alpha_prop_lb_{i}_{j}')
            prob += (alpha[j] <= alpha[i] + M_alpha[j]*(1 - x[i,j]), f'alpha_prop_ub_{i}_{j}')
    for j in requests:
        feasible_rep_predecessors = [i for i in requests if (i,j) in y]
        if feasible_rep_predecessors:
            y_sum = pulp.lpSum(y[i,j] for i in feasible_rep_predecessors)
            prob += (alpha[j] >= z[j] - dist[depot,j] - M_alpha[j]*(1 - y_sum), f'alpha_rep_lb_{j}')
            prob += (alpha[j] <= z[j] - dist[depot,j] + M_alpha[j]*(1 - y_sum), f'alpha_rep_ub_{j}')

    for i in requests:
        if np.isinf(dist[i, depot]):
            continue
        last_indicators = []
        if (i, depot) in x:
            last_indicators.append(x[i, depot])
        last_indicators += [y[i, k] for k in requests if (i, k) in y]
        if last_indicators and i in M_tau:
            last_sum = pulp.lpSum(last_indicators)
            prob += (z[i] + dist[i,depot] - alpha[i] <= T + M_tau[i]*(1 - last_sum),
                     f'shift_duration_{i}')

    for j in requests:
        djd = dist[j, depot] if not np.isinf(dist[j, depot]) else 0.0
        last_indicators = []
        if (j, depot) in x:
            last_indicators.append(x[j, depot])
        last_indicators += [y[j, k] for k in requests if (j, k) in y]
        if last_indicators:
            last_sum = pulp.lpSum(last_indicators)
            prob += (C[j] >= z[j] + djd - M_C[j]*(1 - last_sum), f'C_last_lb_{j}')
            prob += (C[j] <= z[j] + djd + M_C[j]*(1 - last_sum), f'C_last_ub_{j}')
    for (j, k) in direct_arcs:
        if j != depot and k != depot:
            prob += (C[j] >= C[k] - M_C_global*(1 - x[j,k]), f'C_prop_lb_{j}_{k}')
            prob += (C[j] <= C[k] + M_C_global*(1 - x[j,k]), f'C_prop_ub_{j}_{k}')
    for j in requests:
        djd = dist[j, depot] if not np.isinf(dist[j, depot]) else 0.0
        prob += (C[j] >= rt[j] + djd, f'C_earliest_{j}')

    warm_obj = None
    if use_warm_start:
        routes, warm_obj, unserved = greedy_nearest_neighbor(data_dict)
        if verbose and unserved:
            print(f'  Greedy unserved nodes: {unserved}')
        hint_x, hint_y = set(), set()
        for route_nodes, _ in routes:
            full = [depot] + route_nodes + [depot]
            for a, b in zip(full, full[1:]):
                if (a, b) in x: hint_x.add((a, b))
                if (a, b) in y: hint_y.add((a, b))
        for arc in direct_arcs:
            x[arc].setInitialValue(1 if arc in hint_x else 0)
        for arc in replenishment_arcs:
            y[arc].setInitialValue(1 if arc in hint_y else 0)

    # Write CBC log to a temp file so we can parse the gap afterwards
    log_path = Path(tempfile.mktemp(suffix='.log'))
    solver = pulp.PULP_CBC_CMD(msg=1, timeLimit=time_limit, logPath=str(log_path))
    t0 = time.time()
    prob.solve(solver)
    wall_time = time.time() - t0

    status  = pulp.LpStatus[prob.status]
    obj_val = pulp.value(prob.objective) if prob.status in (1, -2) else None

    cbc_obj, lower_bound, mip_gap = None, None, None
    if log_path.exists():
        log_text = log_path.read_text(errors='ignore')
        if verbose:
            print(log_text[-3000:])
        cbc_obj, lower_bound, mip_gap = parse_cbc_gap(log_text)
        log_path.unlink(missing_ok=True)

    if obj_val is None and cbc_obj is not None:
        obj_val = cbc_obj

    truly_optimal = (status == 'Optimal' and (mip_gap is None or mip_gap < 1e-4))

    mip_legs = None
    if prob.status in (1, -2):
        mip_legs = _extract_routes_from_mip(x, y, requests, depot)

    result = {
        'status'      : status,
        'objective'   : round(obj_val,     4) if obj_val     is not None else None,
        'lower_bound' : round(lower_bound, 4) if lower_bound is not None else None,
        'mip_gap_pct' : round(mip_gap * 100, 2) if mip_gap  is not None else None,
        'wall_time'   : round(wall_time, 2),
        'proven_optimal': truly_optimal,
    }
    if mip_legs is not None: result['legs']           = mip_legs
    if warm_obj is not None: result['warm_start_obj'] = round(warm_obj, 4)
    return result


In [8]:
def mip_batch_run(folder_path, time_limit=3600, solver_name='CBC',
                  threads=4, use_warm_start=True, save_csv=True):
    """
    Run the MIP solver on every .txt instance file in a folder and
    optionally save a summary CSV.

    For each instance the function:
      1. Loads and preprocesses the instance.
      2. Calls mip_build_and_solve with the given settings.
      3. Prints a one-line summary and appends the result to a list.

    Exceptions are caught per-file so a single bad instance does not
    abort the entire batch run.

    Returns
    -------
    List of result dicts (one per file), same schema as mip_build_and_solve.
    """
    folder = Path(folder_path)
    files  = sorted(f for f in folder.iterdir() if f.suffix == '.txt')
    if not files:
        print(f'No .txt files found in {folder_path}')
        return []

    results, n_optimal = [], 0
    for fp in files:
        print(f'\n--- {fp.name} ---')
        try:
            raw  = load_instance(fp)
            data = preprocess_instance(raw)
            res  = mip_build_and_solve(data, time_limit=time_limit,
                                       solver_name=solver_name, threads=threads,
                                       verbose=False, use_warm_start=use_warm_start)
            res['file'] = fp.name
            if res['proven_optimal']:
                n_optimal += 1
            gap_str = (f"{res['mip_gap_pct']:.1f}%"
                       if res['mip_gap_pct'] is not None else 'N/A')
            print(f"  Status={res['status']:12s}  Obj={res['objective']}  "
                  f"LB={res['lower_bound']}  Gap={gap_str}  "
                  f"Time={res['wall_time']}s")
        except Exception:
            import traceback; traceback.print_exc()
            res = {'file': fp.name, 'status': 'Error', 'objective': None,
                   'lower_bound': None, 'mip_gap_pct': None,
                   'wall_time': None, 'proven_optimal': False}
        results.append(res)

    print(f'\n=== Summary: {n_optimal}/{len(files)} proven optimal ===')

    if save_csv:
        csv_path = folder / 'results_mip.csv'
        keys = ['file','status','objective','lower_bound',
                'mip_gap_pct','wall_time','proven_optimal','warm_start_obj']
        with open(csv_path, 'w', newline='') as f:
            writer = csv.DictWriter(f, fieldnames=keys, extrasaction='ignore')
            writer.writeheader()
            writer.writerows(results)
        print(f'Results saved to {csv_path}')
    return results


## 4. ALNS (Adaptive Large Neighbourhood Search)

Metaheuristic solver for large instances where the MIP is too slow.

### Solution representation
A solution is a `List[RouteGroup]`, where each `RouteGroup` represents one vehicle.
A vehicle may serve clinics in multiple **legs** separated by mid-shift depot visits
(replenishments).  The RouteGroup dataclass below makes this structure explicit.

### Algorithm outline
1. **Build** an initial solution with the greedy nearest-neighbour heuristic.
2. **Iterate** (up to `max_iter` or `time_limit`):
   a. *Destroy*: remove k clinics using one of three destroy operators (random,
      worst, Shaw).  Operator selection is weighted by past performance.
   b. *Repair*: re-insert removed clinics using greedy or regret-2 insertion.
   c. *Accept/reject*: Simulated Annealing criterion on the objective change.
   d. *Update weights*: reward or penalise the operators used this iteration.
3. **Optional MIP local search**: every `mip_local_freq` iterations, hand a
   subset of routes to the exact MIP solver for local refinement.

### Objective
Same as the MIP: Σ(C[j] − r[j]) computed by `evaluate_solution`.


In [9]:
from dataclasses import dataclass, field
from typing import List

@dataclass
class RouteGroup:
    """
    Represents all work assigned to a single vehicle.

    A vehicle may visit the depot mid-shift to replenish supplies.
    Each uninterrupted stretch between depot visits is called a **leg**.
    Legs are stored as a list of clinic indices (depot index 0 is excluded).

    Example
    -------
    Vehicle serves clinics [1, 2], returns to depot, then serves [3, 4]:
        legs = [[1, 2], [3, 4]]

    The implicit route sequence is:
        depot → 1 → 2 → depot → 3 → 4 → depot
    """
    legs: List[List[int]] = field(default_factory=list)

    def all_jobs(self) -> List[int]:
        """Return all clinic indices visited by this vehicle, in order."""
        return [j for leg in self.legs for j in leg]

    def n_replenishments(self) -> int:
        """Number of mid-shift depot visits = number of legs minus one."""
        return max(0, len(self.legs) - 1)

    def is_empty(self) -> bool:
        """True if no clinics are assigned to this vehicle."""
        return all(len(leg) == 0 for leg in self.legs)

    def copy(self) -> "RouteGroup":
        """Deep copy: legs and their contents are independent of the original."""
        return RouteGroup(legs=[list(leg) for leg in self.legs])


### 4.1 Solution Evaluation


In [10]:
def _simulate_group(grp: RouteGroup, data_dict: dict, return_details=False):
    """
    Simulate ONE vehicle across all of its legs on one continuous clock.

    A RouteGroup represents one physical vehicle.  If its legs are
        [[1, 2], [3, 4]]
    the actual sequence is
        0 -> 1 -> 2 -> 0 -> 3 -> 4 -> 0
    and the second leg cannot start before the first leg has returned to depot.

    The first departure may be delayed so that the vehicle reaches the first
    clinic just in time.  This follows Rocha et al.'s shift definition: the
    shift starts when the vehicle leaves the depot for its first trip and ends
    when it returns after its last trip.

    Objective contribution is the corrected completion time
        sum_j (C[j] - r[j]),
    where C[j] is the depot-return time of the leg carrying request j.

    Returns
    -------
    If return_details=False:
        (cost, feasible)
    If return_details=True:
        (cost, feasible, details)
    """
    dist  = data_dict['dist_matrix']
    rt    = data_dict['ready_times']
    dd    = data_dict['due_dates']
    T     = data_dict['max_tau']
    depot = 0
    eps   = 1e-6

    vehicle_clock  = 0.0
    first_departure = None
    total = 0.0
    arrivals = {}
    completions = {}
    leg_returns = []
    leg_departures = []

    for leg in grp.legs:
        if not leg:
            continue

        first = leg[0]
        if np.isinf(dist[depot, first]):
            result = (np.inf, False)
            return (*result, None) if return_details else result

        # The vehicle is already at the depot after the preceding leg.
        # It may wait at the depot before starting the next trip.
        leg_depart = max(vehicle_clock, 0.0, rt[first] - dist[depot, first])
        if first_departure is None:
            first_departure = leg_depart

        cur_time, prev = leg_depart, depot
        for j in leg:
            if np.isinf(dist[prev, j]):
                result = (np.inf, False)
                return (*result, None) if return_details else result

            arr = max(cur_time + dist[prev, j], rt[j])
            if arr > dd[j] + eps:
                result = (np.inf, False)
                return (*result, None) if return_details else result

            arrivals[j] = arr
            cur_time, prev = arr, j

        if np.isinf(dist[prev, depot]):
            result = (np.inf, False)
            return (*result, None) if return_details else result

        return_time = cur_time + dist[prev, depot]

        # d0 is the latest return to the laboratory.  Since leg return times are
        # nondecreasing, checking every depot return is equivalent to checking
        # the final one and catches infeasibility as early as possible.
        if return_time > dd[depot] + eps:
            result = (np.inf, False)
            return (*result, None) if return_details else result

        # FULL vehicle shift: first departure -> current/final depot return.
        if first_departure is not None and return_time - first_departure > T + eps:
            result = (np.inf, False)
            return (*result, None) if return_details else result

        for j in leg:
            completions[j] = return_time
            total += return_time - rt[j]

        leg_departures.append(leg_depart)
        leg_returns.append(return_time)
        vehicle_clock = return_time

    details = {
        'arrivals': arrivals,
        'completions': completions,
        'leg_departures': leg_departures,
        'leg_returns': leg_returns,
        'first_departure': first_departure,
        'final_return': vehicle_clock if first_departure is not None else None,
        'shift_duration': (vehicle_clock - first_departure)
                          if first_departure is not None else 0.0,
    }
    return (total, True, details) if return_details else (total, True)


def evaluate_solution(groups: List[RouteGroup], data_dict: dict) -> float:
    """
    Compute corrected total completion time sum_j (C[j] - r[j]).

    IMPORTANT: each RouteGroup is one vehicle and all of its legs are simulated
    sequentially on the same clock.  Depot visits between legs are replenishment
    visits, not new vehicles and not new time origins.

    Returns np.inf if any vehicle route violates a time window, depot-return
    deadline, full-shift duration, arc feasibility, or fleet-size constraint.
    """
    K = data_dict['max_vehicles']

    nonempty_groups = [g for g in groups if not g.is_empty()]
    if len(nonempty_groups) > K:
        return np.inf

    total = 0.0
    for grp in nonempty_groups:
        cost, feasible = _simulate_group(grp, data_dict)
        if not feasible:
            return np.inf
        total += cost

    return total


def is_feasible(groups, data_dict):
    """Return True iff the solution satisfies all modelled constraints."""
    return not np.isinf(evaluate_solution(groups, data_dict))



### 4.2 Destroy Operators


In [11]:
def _job_location(groups: List[RouteGroup]):
    """Build a reverse index: {clinic_index: (group_idx, leg_idx)}."""
    loc = {}
    for gi, grp in enumerate(groups):
        for li, leg in enumerate(grp.legs):
            for j in leg:
                loc[j] = (gi, li)
    return loc


def _remove_jobs(groups: List[RouteGroup], to_remove: set) -> List[RouteGroup]:
    """
    Remove a set of clinics from the solution.

    After removal, empty legs are pruned.  Each RouteGroup retains at least one
    leg (possibly empty) so the group structure (vehicle count) is preserved for
    subsequent repair.
    """
    new_groups = []
    for grp in groups:
        new_legs = [
            [j for j in leg if j not in to_remove]
            for leg in grp.legs
        ]
        # Keep at least one (possibly empty) leg to preserve the group slot
        new_legs = [leg for leg in new_legs if leg] or [[]]
        new_groups.append(RouteGroup(legs=new_legs))
    return new_groups


def random_removal(groups: List[RouteGroup], data_dict, k, rng):
    """
    Randomly remove k clinics from the solution (uniform sampling).
    Simplest destroy operator; provides exploration diversity.
    """
    all_jobs = [j for grp in groups for j in grp.all_jobs()]
    if not all_jobs:
        return [g.copy() for g in groups], []
    removed = rng.sample(all_jobs, min(k, len(all_jobs)))
    return _remove_jobs(groups, set(removed)), removed


def worst_removal(groups: List[RouteGroup], data_dict, k, rng):
    """
    Remove clinics with the largest current corrected completion contribution.

    Timing is taken from the full sequential RouteGroup simulation, so service
    and completion times of later legs include all earlier depot-return trips.
    """
    rt = data_dict['ready_times']
    costs = []

    for grp in groups:
        _, feasible, details = _simulate_group(grp, data_dict, return_details=True)
        if not feasible:
            continue
        for j, Cj in details['completions'].items():
            costs.append((Cj - rt[j], j))

    costs.sort(key=lambda x: -x[0] - rng.uniform(0, 1e-6))
    removed = [j for _, j in costs[:k]]
    return _remove_jobs(groups, set(removed)), removed


def shaw_removal(groups: List[RouteGroup], data_dict, k, rng):
    """
    Shaw removal using spatial distance and ACTUAL sequential service times.
    """
    dist = data_dict['dist_matrix']
    arrival = {}

    for grp in groups:
        _, feasible, details = _simulate_group(grp, data_dict, return_details=True)
        if not feasible:
            continue
        arrival.update(details['arrivals'])

    all_jobs = list(arrival.keys())
    if not all_jobs:
        return [g.copy() for g in groups], []

    seed = rng.choice(all_jobs)
    removed = [seed]

    for _ in range(min(k - 1, len(all_jobs) - 1)):
        ref = rng.choice(removed)
        candidates = [j for j in all_jobs if j not in removed]
        if not candidates:
            break
        best_j = min(
            candidates,
            key=lambda j: dist[ref, j] + abs(arrival[j] - arrival[ref])
        )
        removed.append(best_j)

    return _remove_jobs(groups, set(removed)), removed



### 4.3 Repair Operators


In [12]:
def _leg_cost(leg: List[int], data_dict: dict):
    """
    Simulate a single leg and return (return_time, depart_time, feasible).

    Kept as a simple full-resimulation helper (used by evaluate_solution-style
    callers that just need a final feasibility/cost check, not incremental
    reuse). The hot path used during insertion search is _leg_arrival_profile
    + _insertion_cost_fast below.
    """
    dist  = data_dict['dist_matrix']
    rt    = data_dict['ready_times']
    dd    = data_dict['due_dates']
    T     = data_dict['max_tau']
    depot = 0

    cur_time, prev = 0.0, depot
    depart_time = None

    for node in leg:
        if np.isinf(dist[prev, node]):
            return None, None, False
        arr = max(cur_time + dist[prev, node], rt[node])
        if arr > dd[node]:
            return None, None, False
        if depart_time is None:
            depart_time = arr - dist[depot, node]
        cur_time, prev = arr, node

    if np.isinf(dist[prev, depot]):
        return None, None, False
    return_time = cur_time + dist[prev, depot]

    if depart_time is not None and return_time - depart_time > T + 1e-6:
        return None, None, False

    return return_time, depart_time, True


def _leg_arrival_profile(leg: List[int], data_dict: dict):
    """
    Full simulation of a leg that ALSO returns the per-node arrival times.

    This is the building block for the "incremental cost update" optimisation:
    profiling showed that re-simulating a leg's full timeline from the depot
    on EVERY candidate insertion position was the single largest cost in the
    ALNS main loop (~68% of total runtime, dominated by _leg_cost calls).

    Nodes before an insertion point are never affected by the insertion
    (time propagation is strictly left-to-right), so callers can reuse
    arrival_times[pos-1] as the starting point instead of resimulating the
    whole prefix every time.

    Returns (arrival_times, depart_time, return_time, feasible).
    arrival_times[i] is the arrival time at leg[i].
    """
    dist  = data_dict['dist_matrix']
    rt    = data_dict['ready_times']
    dd    = data_dict['due_dates']
    T     = data_dict['max_tau']
    depot = 0

    cur_time, prev = 0.0, depot
    depart_time = None
    arrival_times = []

    for node in leg:
        if np.isinf(dist[prev, node]):
            return None, None, None, False
        arr = max(cur_time + dist[prev, node], rt[node])
        if arr > dd[node]:
            return None, None, None, False
        if depart_time is None:
            depart_time = arr - dist[depot, node]
        arrival_times.append(arr)
        cur_time, prev = arr, node

    if np.isinf(dist[prev, depot]):
        return None, None, None, False
    return_time = cur_time + dist[prev, depot]

    if depart_time is not None and return_time - depart_time > T + 1e-6:
        return None, None, None, False

    return arrival_times, depart_time, return_time, True


def _insertion_cost_fast(grp: RouteGroup, leg_idx: int, old_leg: List[int], pos: int,
                         j: int, data_dict: dict,
                         arr_times, depart_time_old, sum_rt_leg, return_time_old):
    """
    Incremental replacement for the old _insertion_cost.

    Reuses the leg's cached arrival profile (computed ONCE per leg by the
    caller, not once per candidate position as before) and only resimulates
    the suffix starting at `pos` — the inserted node j and everything after
    it — instead of the entire old leg AND the entire new leg from the depot.

    Because every node in a leg shares the same completion time C[j] =
    return_time, the objective delta can be computed from just
    (old return_time, new return_time, leg length, sum of ready times) in
    O(1), without re-summing per-node contributions.

    Returns (delta_obj, new_grp) or (inf, None) if insertion is infeasible.
    """
    dist  = data_dict['dist_matrix']
    rt    = data_dict['ready_times']
    dd    = data_dict['due_dates']
    T     = data_dict['max_tau']
    depot = 0

    if pos == 0:
        cur_time, prev = 0.0, depot
        depart_time = None
    else:
        cur_time, prev = arr_times[pos - 1], old_leg[pos - 1]
        depart_time = depart_time_old

    for node in [j] + old_leg[pos:]:
        if np.isinf(dist[prev, node]):
            return np.inf, None
        arr = max(cur_time + dist[prev, node], rt[node])
        if arr > dd[node]:
            return np.inf, None
        if depart_time is None:
            depart_time = arr - dist[depot, node]
        cur_time, prev = arr, node

    if np.isinf(dist[prev, depot]):
        return np.inf, None
    return_time_new = cur_time + dist[prev, depot]

    if depart_time is not None and return_time_new - depart_time > T + 1e-6:
        return np.inf, None

    old_contribution = len(old_leg) * return_time_old - sum_rt_leg
    new_contribution = (len(old_leg) + 1) * return_time_new - (sum_rt_leg + rt[j])

    new_grp = grp.copy()
    new_grp.legs[leg_idx] = old_leg[:pos] + [j] + old_leg[pos:]
    return new_contribution - old_contribution, new_grp


def _new_leg_cost(j: int, data_dict: dict):
    """
    Cost of opening a brand-new single-clinic leg for clinic j.

    Used when no existing leg can feasibly accommodate j (options B and C in
    _all_insertion_options).  The vehicle departs depot at the latest moment
    that still allows it to arrive exactly at ready_time[j].

    Returns (obj_contribution, feasible).
    """
    dist  = data_dict['dist_matrix']
    rt    = data_dict['ready_times']
    dd    = data_dict['due_dates']
    depot = 0

    if np.isinf(dist[depot, j]) or np.isinf(dist[j, depot]):
        return np.inf, False
    arr = max(dist[depot, j], rt[j])
    if arr > dd[j]:
        return np.inf, False
    return_time = arr + dist[j, depot]
    return return_time - rt[j], True


def _build_neighbor_lists(data_dict: dict, k: int = 20):
    """
    Candidate list: for each clinic j, precompute its k spatially nearest
    other clinics (using the arc-pruned distance matrix).

    Used to prune _all_insertion_options / _group_insertion_options: a leg
    that contains none of j's nearest neighbours is skipped outright, since
    inserting j into a leg it is geographically unrelated to is essentially
    never competitive. This cuts the number of full insertion evaluations
    from O(n) to roughly O(k) per removed clinic, without touching the
    objective/feasibility logic itself — it only restricts which candidates
    are tried, so any accepted move is exactly as valid as before.
    """
    dist = data_dict['dist_matrix']
    N    = data_dict['num_nodes']
    neighbor_lists = {}
    for j in range(1, N):
        row = dist[j]
        order = np.argsort(row)
        cands = [int(i) for i in order
                 if i != j and i != 0 and np.isfinite(row[i])]
        neighbor_lists[j] = set(cands[:k])
    return neighbor_lists

In [13]:
def _group_insertion_options(gi, grp: RouteGroup, j: int, data_dict: dict):
    """
    Enumerate feasible insertions of j into ONE RouteGroup.

    Cost is evaluated at RouteGroup level rather than leg level.  This is
    necessary for a multi-trip vehicle because changing an early leg can delay
    every later leg and can change the full-shift feasibility.

    Options:
      A) insert j at any position of an existing leg;
      B) append a new replenishment leg [j] to this same vehicle.
    """
    options = []
    neighbor_lists = data_dict.get('neighbor_lists')
    neighbors_j = neighbor_lists.get(j) if neighbor_lists else None

    old_cost, old_ok = _simulate_group(grp, data_dict)
    if not old_ok:
        return options

    # A) Insert into existing leg.
    for li, leg in enumerate(grp.legs):
        if leg and neighbors_j is not None and not any(n in neighbors_j for n in leg):
            continue

        for pos in range(len(leg) + 1):
            new_grp = grp.copy()
            new_grp.legs[li] = leg[:pos] + [j] + leg[pos:]
            new_cost, ok = _simulate_group(new_grp, data_dict)
            if not ok:
                continue
            delta = new_cost - old_cost

            def make_apply_A(gi=gi, new_grp=new_grp):
                def apply(gs):
                    gs = [g.copy() for g in gs]
                    gs[gi] = new_grp.copy()
                    return gs
                apply.gi = gi
                return apply

            options.append((delta, make_apply_A()))

    # B) Add a new trip after the existing trips of the same vehicle.
    if not grp.is_empty():
        new_grp = grp.copy()
        new_grp.legs.append([j])
        new_cost, ok = _simulate_group(new_grp, data_dict)
        if ok:
            delta = new_cost - old_cost

            def make_apply_B(gi=gi, new_grp=new_grp):
                def apply(gs):
                    gs = [g.copy() for g in gs]
                    gs[gi] = new_grp.copy()
                    return gs
                apply.gi = gi
                return apply

            options.append((delta, make_apply_B()))

    return options


def _new_group_option(groups: List[RouteGroup], j: int, data_dict: dict):
    """Option C: deploy one additional vehicle for a single-clinic trip."""
    K = data_dict['max_vehicles']
    nonempty_count = sum(not g.is_empty() for g in groups)
    if nonempty_count >= K:
        return None

    new_grp = RouteGroup(legs=[[j]])
    contrib_j, ok = _simulate_group(new_grp, data_dict)
    if not ok:
        return None

    def apply_C(gs, j=j):
        gs = [g.copy() for g in gs]
        gs.append(RouteGroup(legs=[[j]]))
        return gs
    return (contrib_j, apply_C)


def _all_insertion_options(groups: List[RouteGroup], j: int, data_dict: dict):
    """Enumerate every feasible A/B/C insertion option for clinic j."""
    options = []
    for gi, grp in enumerate(groups):
        options.extend(_group_insertion_options(gi, grp, j, data_dict))
    c_opt = _new_group_option(groups, j, data_dict)
    if c_opt is not None:
        options.append(c_opt)
    return options


def greedy_insert(groups: List[RouteGroup], removed: list,
                  data_dict: dict, rng) -> List[RouteGroup]:
    """Cheapest insertion using full RouteGroup-level timing/cost evaluation."""
    groups = [g.copy() for g in groups]
    removed = list(removed)
    rng.shuffle(removed)

    for j in removed:
        opts = _all_insertion_options(groups, j, data_dict)
        if not opts:
            # No feasible insertion under the current partial solution.
            # Return an intentionally infeasible candidate; the ALNS caller will
            # reject it rather than silently exceeding the fleet/shift limits.
            groups.append(RouteGroup(legs=[[j]]))
            continue
        opts.sort(key=lambda x: x[0])
        _, apply = opts[0]
        groups = apply(groups)

    return [g for g in groups if not g.is_empty()]


def regret2_insert(groups: List[RouteGroup], removed: list,
                   data_dict: dict, rng) -> List[RouteGroup]:
    """
    Regret-2 insertion.  Candidate costs are recomputed from full RouteGroup
    schedules so downstream legs and full-shift duration remain correct.
    """
    groups = [g.copy() for g in groups]
    unplaced = list(removed)

    while unplaced:
        best_j = None
        best_regret = -np.inf
        best_apply = None

        for j in unplaced:
            opts = _all_insertion_options(groups, j, data_dict)
            if not opts:
                regret = -np.inf
                apply_fn = None
            else:
                opts.sort(key=lambda x: x[0])
                apply_fn = opts[0][1]
                regret = (opts[1][0] - opts[0][0]) if len(opts) > 1 else np.inf

            if regret > best_regret:
                best_regret = regret
                best_j = j
                best_apply = apply_fn

        if best_j is None:
            # No feasible placement for any remaining request.
            # Keep the candidate structurally complete; final feasibility check
            # in the ALNS loop will reject it.
            best_j = unplaced[0]

        unplaced.remove(best_j)
        if best_apply is None:
            groups.append(RouteGroup(legs=[[best_j]]))
        else:
            groups = best_apply(groups)

    return [g for g in groups if not g.is_empty()]



### 4.4 Adaptive Weight Manager


In [14]:
class AdaptiveWeights:
    """
    Roulette-wheel operator selector with segment-based weight updates.

    Each operator starts with weight 1.0.  Within each segment of
    `segment_size` iterations the operator accumulates a score based on
    the quality of solutions it produced:

        sigma[0] = reward for finding a new global best
        sigma[1] = reward for improving the current solution (not global best)
        sigma[2] = reward for a solution that is accepted (but not improving)

    At the end of each segment the weight is updated with exponential smoothing:
        w_new = decay * w_old + (1 - decay) * avg_score_this_segment

    The `decay` parameter balances long-term reputation (high decay) vs.
    responsiveness to recent performance (low decay).

    Selection uses roulette wheel (fitness-proportionate sampling):
    an operator's probability of being selected equals its weight divided by
    the total weight of all operators.
    """
    def __init__(self, operators, sigma1=3, sigma2=2, sigma3=1,
                 decay=0.8, segment_size=100):
        self.operators    = operators
        self.sigma        = [sigma1, sigma2, sigma3]
        self.decay        = decay
        self.segment_size = segment_size
        self.weights      = {op: 1.0 for op in operators}
        self._scores      = {op: 0.0 for op in operators}
        self._uses        = {op: 0   for op in operators}
        self._iter        = 0

    def select(self, rng):
        """Sample one operator proportional to its current weight."""
        ops   = self.operators
        w     = [self.weights[op] for op in ops]
        total = sum(w)
        r     = rng.uniform(0, total)
        cum   = 0.0
        for op, wi in zip(ops, w):
            cum += wi
            if r <= cum:
                return op
        return ops[-1]   # fallback for floating-point edge cases

    def update(self, op, reward_tier):
        """
        Record that `op` was used and produced a result of `reward_tier`:
            0 = new global best
            1 = improved current solution (but not global best)
            2 = accepted a worse solution (or rejected)
        """
        self._scores[op] += self.sigma[reward_tier]
        self._uses[op]   += 1
        self._iter       += 1
        if self._iter % self.segment_size == 0:
            self._flush_weights()

    def _flush_weights(self):
        """Apply exponential smoothing at the end of each segment and reset accumulators."""
        for op in self.operators:
            if self._uses[op] > 0:
                score = self._scores[op] / self._uses[op]
                self.weights[op] = (self.decay * self.weights[op] +
                                    (1 - self.decay) * score)
            self._scores[op] = 0.0
            self._uses[op]   = 0


### 4.5 Solution Initializer


In [15]:
def build_initial_solution(data_dict: dict):
    """
    Construct the ALNS starting solution using the greedy nearest-neighbour heuristic.

    Each greedy route becomes a RouteGroup with a single leg (no replenishment).
    Clinics that the greedy could not assign ('unvisited') are each placed in their
    own single-clinic RouteGroup, provided the direct depot→j→depot trip is feasible.
    This ensures the initial solution is complete (all clinics assigned), which is
    required for the ALNS main loop to operate correctly.
    """
    routes_raw, _, unvisited = greedy_nearest_neighbor(data_dict)
    groups = [RouteGroup(legs=[list(r)]) for r, _ in routes_raw]

    dist  = data_dict['dist_matrix']
    rt    = data_dict['ready_times']
    dd    = data_dict['due_dates']
    depot = 0
    for j in unvisited:
        if not np.isinf(dist[depot, j]) and not np.isinf(dist[j, depot]):
            arr = max(dist[depot, j], rt[j])
            if arr <= dd[j]:
                groups.append(RouteGroup(legs=[[j]]))

    return groups


### 4.6 Request-based restricted MIP intensification

The earlier route-based restricted MIP was too narrow: reoptimizing one or two complete RouteGroups rarely improved the incumbent. The final hybrid design therefore uses a **request-induced exact neighborhood**.

At each exact-intensification point:

1. Evaluate the current best ALNS solution.
2. Rank requests by their corrected completion-time contribution \(C_j-r_j\).
3. Select the highest-cost requests.
4. Identify the vehicle groups containing those requests.
5. Jointly reoptimize **all requests currently assigned to the affected vehicles** with the strict Rocha MIP.
6. Preserve replenishment-trip grouping when mapping the exact solution back to ALNS.
7. Accept the restricted-MIP result only if it strictly improves the current global best.

This allows cross-route reassignment among several affected vehicles, while keeping the exact subproblem substantially smaller than the full 100-request instance.

**Final tested production parameters**

- exact call frequency: every **500 ALNS iterations**;
- high-cost request target: **8 requests**;
- affected vehicles: at least **3**, at most **5**;
- exact time budget per call: **20 seconds**;
- Gurobi threads inside restricted MIP: **4**;
- exact step operates on `best_groups`, not on the current SA state;
- no ALNS random numbers are consumed by neighborhood selection, so the stochastic operator stream is not changed unless the exact step actually improves the incumbent.

The parameter-development process and operator diagnostics are isolated in Section 6 and are not executed by a normal `Run All`.


In [16]:
def _request_contributions(groups, data_dict):
    """
    Return (request, corrected completion contribution, group index), sorted
    from highest to lowest contribution.
    """
    rt = data_dict['ready_times']
    scored = []

    for gi, grp in enumerate(groups):
        cost, feasible, details = _simulate_group(grp, data_dict, return_details=True)
        if not feasible or details is None:
            continue

        for j, completion in details['completions'].items():
            scored.append((int(j), float(completion - rt[j]), int(gi)))

    scored.sort(key=lambda x: x[1], reverse=True)
    return scored


def _select_request_induced_groups(groups, data_dict,
                                   q_requests=8,
                                   min_groups=3,
                                   max_groups=5):
    """
    Select high-cost requests first.  The vehicles containing those requests
    define the restricted exact neighborhood.
    """
    ranked = _request_contributions(groups, data_dict)

    selected_requests = []
    affected_groups = []

    for j, score, gi in ranked:
        if gi not in affected_groups and len(affected_groups) >= max_groups:
            continue

        selected_requests.append(j)
        if gi not in affected_groups:
            affected_groups.append(gi)

        if len(selected_requests) >= q_requests and len(affected_groups) >= min_groups:
            break

    if len(affected_groups) < min_groups:
        for j, score, gi in ranked:
            if gi not in affected_groups:
                affected_groups.append(gi)
                if j not in selected_requests:
                    selected_requests.append(j)
            if len(affected_groups) >= min_groups:
                break

    return selected_requests, affected_groups[:max_groups], ranked


def request_based_mip_local_search(groups, data_dict,
                                   q_requests=8,
                                   min_groups=3,
                                   max_groups=5,
                                   time_limit=20,
                                   threads=4,
                                   verbose=False):
    """
    Jointly reoptimize the vehicle groups induced by the worst requests.

    The subproblem is solved with the same strict Rocha Gurobi formulation used
    for the full exact benchmark.  A candidate replaces the incumbent only when
    the complete ALNS objective strictly improves.
    """
    current_total = evaluate_solution(groups, data_dict)

    selected_requests, affected, ranked = _select_request_induced_groups(
        groups, data_dict,
        q_requests=q_requests,
        min_groups=min_groups,
        max_groups=max_groups
    )

    info = {
        'selected_requests': selected_requests,
        'affected_groups': affected,
        'sub_jobs': [],
        'mip_status': 'Skipped',
        'mip_obj': None,
        'candidate_obj': current_total,
        'improved': False,
        'mip_time': 0.0,
    }

    if not affected:
        return groups, current_total, info

    sub_jobs = []
    for gi in affected:
        sub_jobs.extend(groups[gi].all_jobs())
    sub_jobs = list(dict.fromkeys(sub_jobs))
    info['sub_jobs'] = sub_jobs

    sub_nodes = [0] + sub_jobs
    sub_data = {
        'coords': np.asarray(data_dict['coords'])[sub_nodes],
        'ready_times': np.asarray(data_dict['ready_times'])[sub_nodes],
        'due_dates': np.asarray(data_dict['due_dates'])[sub_nodes],
        'max_vehicles': len(affected),
        'max_tau': data_dict['max_tau'],
        'num_nodes': len(sub_nodes),
        'dist_matrix': np.asarray(data_dict['dist_matrix'])[np.ix_(sub_nodes, sub_nodes)],
    }

    t0 = time.time()
    mip_res = mip_build_and_solve(
        sub_data,
        time_limit=time_limit,
        solver_name='GUROBI',
        threads=threads,
        verbose=False,
        use_warm_start=False
    )
    info['mip_time'] = time.time() - t0
    info['mip_status'] = mip_res.get('status')
    info['mip_obj'] = mip_res.get('objective')

    mip_groups = mip_res.get('route_groups')
    if not mip_groups:
        return groups, current_total, info

    remapped_groups = []
    for vehicle_legs in mip_groups:
        mapped_legs = []
        for leg in vehicle_legs:
            mapped_legs.append([sub_nodes[k] for k in leg])
        remapped_groups.append(RouteGroup(legs=mapped_legs))

    while len(remapped_groups) < len(affected):
        remapped_groups.append(RouteGroup(legs=[]))

    if len(remapped_groups) > len(affected):
        info['mip_status'] = 'Rejected: too many route groups'
        return groups, current_total, info

    candidate = [g.copy() for g in groups]
    for slot, gi in enumerate(affected):
        candidate[gi] = remapped_groups[slot]

    candidate_obj = evaluate_solution(candidate, data_dict)
    info['candidate_obj'] = candidate_obj
    info['improved'] = (
        np.isfinite(candidate_obj)
        and candidate_obj < current_total - 1e-6
    )

    if verbose:
        print(
            f"  [Restricted MIP] groups={affected}, jobs={len(sub_jobs)}, "
            f"{current_total:.4f} -> {candidate_obj:.4f}, "
            f"status={info['mip_status']}, time={info['mip_time']:.2f}s"
        )

    if info['improved']:
        return candidate, candidate_obj, info

    return groups, current_total, info


In [17]:
def _extract_routes_from_mip(x, y, requests, depot):
    """
    Reconstruct legs from PuLP variable values after a CBC solve.

    Logic mirrors _extract_routes_from_mip_gurobi; only the value-access API
    differs (pulp.value(var) instead of var.X).

    Walk each route starting from nodes that the depot dispatches to directly
    (x[depot,j] = 1).  At each clinic:
      - If the clinic is followed by a replenishment arc (y[i,k] = 1 for some k),
        close the current leg and begin a new one.
      - If followed by a direct arc to depot, the route ends.
      - Otherwise, follow the direct arc to the next clinic.

    Returns List[List[int]] — one inner list per leg (depot index excluded).
    """
    # Build successor maps from active arcs
    succ_x = {}
    for (i, j), var in x.items():
        if pulp.value(var) > 0.5:
            succ_x[i] = j

    replenish_after = set()   # clinics after which the vehicle visits depot
    succ_y = {}
    for (i, j), var in y.items():
        if pulp.value(var) > 0.5:
            replenish_after.add(i)
            succ_y[i] = j

    legs = []
    route_starts = [j for j in requests
                    if (depot, j) in x and pulp.value(x[depot, j]) > 0.5]

    visited = set()
    for start in route_starts:
        leg = []
        cur = start
        while cur != depot and cur not in visited:
            visited.add(cur)
            leg.append(cur)
            if cur in replenish_after:
                legs.append(leg)      # end of this leg
                leg = []
                cur = succ_y.get(cur, depot)
            else:
                cur = succ_x.get(cur, depot)
        if leg:
            legs.append(leg)

    return legs


### 4.7 Main ALNS solver

The main ALNS combines:

- greedy initialization;
- three destroy operators: random, worst and Shaw removal;
- two repair operators: greedy and regret-2 insertion;
- adaptive roulette-wheel operator selection;
- simulated-annealing acceptance;
- continuous multi-trip vehicle timing;
- optional request-based restricted-MIP intensification.

The exact component is disabled by setting `mip_local_freq=0`. With the final hybrid setting, it is called every 500 iterations using the request-induced neighborhood defined in Section 4.6.

For diagnostics, the solver also returns operator usage counts, new-global-best counts, current-solution improvement counts, and restricted-MIP call information. These statistics do not change the search logic and are used below for the pure-ALNS versus hybrid-ALNS component analysis.


In [18]:
def alns_build_and_solve(data_dict,
               max_iter=2000,
               time_limit=300,
               removal_min=2,
               removal_max=6,
               t_init=None,
               cooling=0.9975,
               segment_size=100,
               seed=42,
               verbose=True,
               mip_local_freq=0,
               mip_local_time=20,
               mip_local_q=8,
               mip_local_min_groups=3,
               mip_local_max_groups=5,
               candidate_list_size=20):
    """
    Main ALNS solver for CDSP with Simulated Annealing acceptance and optional
    MIP local search refinement.

    Algorithm flow
    --------------
    1. Build initial solution with greedy nearest-neighbour.
    2. For each iteration (up to max_iter or time_limit):
       a. Optionally run request-based restricted MIP on the current best solution.
       b. Select destroy operator d_op (roulette wheel, adaptive weights).
       c. Select repair operator r_op (roulette wheel, adaptive weights).
       d. Remove k clinics (k sampled uniformly from [removal_min, removal_max],
          capped at N//3 to avoid destroying too much on large instances).
       e. Re-insert removed clinics with r_op.
       f. Verify all clinics are present (skip if repair was incomplete).
       g. Accept/reject the candidate via Simulated Annealing:
            accepted if delta < 0  OR  random() < exp(−delta / T)
       h. Update operator scores and weights.
       i. Cool the temperature: T *= cooling.
    3. Return best solution found.

    Simulated Annealing temperature
    --------------------------------
    t_init defaults to max(greedy_obj * 0.05, 1.0), calibrated so that a 5%
    deterioration is accepted ~37% of the time at the start.  The cooling factor
    is applied every iteration; at cooling=0.9975 the temperature halves every
    ~277 iterations.

    Reward tiers for operator weight updates
    -----------------------------------------
    0 (sigma1) : candidate is a new global best
    1 (sigma2) : candidate improves current solution but not global best
    2 (sigma3) : candidate is accepted (worse or equal) or rejected

    Parameters
    ----------
    data_dict        : preprocessed instance dict
    max_iter         : maximum number of ALNS iterations
    time_limit       : wall-clock cutoff in seconds
    removal_min/max  : bounds on k (number of clinics removed per iteration)
    t_init           : initial SA temperature (None = auto)
    cooling          : multiplicative cooling factor per iteration
    segment_size     : number of iterations per adaptive weight update segment
    seed             : RNG seed for reproducibility
    verbose          : print progress every 200 iterations
    mip_local_freq   : run MIP local search every this many iterations (0 = disabled)
    mip_local_time   : time budget (seconds) per MIP local search call
    mip_local_q       : number of high-cost requests used to induce the exact neighborhood
    mip_local_min_groups : minimum number of affected vehicle groups
    mip_local_max_groups : maximum number of affected vehicle groups
    candidate_list_size : size of the spatial candidate list used to prune
                          insertion search (see _build_neighbor_lists);
                          set to a large number (>= num_nodes) to disable
                          pruning and fall back to exhaustive search.

    Returns
    -------
    dict: objective, routes (flat list of legs), groups (List[RouteGroup]),
          n_replenishments, wall_time, iterations,
          destroy_weights, repair_weights
    """
    rng = random.Random(seed)
    t0  = time.time()

    # Candidate list: precomputed once per instance, reused by every destroy/
    # repair call via data_dict['neighbor_lists']. See _build_neighbor_lists.
    data_dict['neighbor_lists'] = _build_neighbor_lists(data_dict, k=candidate_list_size)

    current_groups = build_initial_solution(data_dict)
    current_obj    = evaluate_solution(current_groups, data_dict)
    best_groups    = [g.copy() for g in current_groups]
    best_obj       = current_obj

    if t_init is None:
        t_init = max(current_obj * 0.05, 1.0)
    temperature = t_init

    if verbose:
        print(f"ALNS start  | Greedy obj (corrected) = {current_obj:.4f}")
        print(f"            | t_init={t_init:.4f}  cooling={cooling}  "
              f"max_iter={max_iter}  time_limit={time_limit}s")
        if mip_local_freq > 0:
            print(f"            | Restricted MIP: every {mip_local_freq} iters, "
                  f"q={mip_local_q}, groups={mip_local_min_groups}-{mip_local_max_groups}, "
                  f"{mip_local_time}s budget")

    destroy_ops = ['random', 'worst', 'shaw']
    repair_ops  = ['greedy', 'regret2']
    d_weights   = AdaptiveWeights(destroy_ops, segment_size=segment_size)
    r_weights   = AdaptiveWeights(repair_ops,  segment_size=segment_size)
    destroy_fn  = {'random': random_removal, 'worst': worst_removal, 'shaw': shaw_removal}
    repair_fn   = {'greedy': greedy_insert,  'regret2': regret2_insert}

    # Diagnostics only; these counters do not affect the search.
    destroy_usage = {op: 0 for op in destroy_ops}
    repair_usage = {op: 0 for op in repair_ops}
    destroy_new_best = {op: 0 for op in destroy_ops}
    repair_new_best = {op: 0 for op in repair_ops}
    destroy_improve_current = {op: 0 for op in destroy_ops}
    repair_improve_current = {op: 0 for op in repair_ops}
    mip_details = []

    for it in range(1, max_iter + 1):
        if time.time() - t0 > time_limit:
            break

        # ── Optional request-based restricted exact intensification ───────────
        if mip_local_freq > 0 and it % mip_local_freq == 0 and best_groups:
            improved_groups, improved_obj, mip_info = request_based_mip_local_search(
                best_groups,
                data_dict,
                q_requests=mip_local_q,
                min_groups=mip_local_min_groups,
                max_groups=mip_local_max_groups,
                time_limit=mip_local_time,
                threads=4,
                verbose=verbose
            )
            mip_info['iteration'] = it
            mip_info['before_best'] = best_obj
            mip_info['after_best'] = improved_obj
            mip_details.append(mip_info)

            if improved_obj < best_obj - 1e-6:
                best_groups = current_groups = [g.copy() for g in improved_groups]
                best_obj = current_obj = improved_obj
                if verbose:
                    print(f"  [MIP] Iter {it}: new best = {best_obj:.4f}")

        # ── Select operators ──────────────────────────────────────────────────
        d_op = d_weights.select(rng)
        r_op = r_weights.select(rng)
        destroy_usage[d_op] += 1
        repair_usage[r_op] += 1

        # Removal size k: scale with instance size, capped to avoid over-destruction
        N = data_dict['num_nodes'] - 1
        k = rng.randint(removal_min, min(removal_max, max(removal_min, N // 3)))

        # ── Destroy → Repair ──────────────────────────────────────────────────
        partial, removed = destroy_fn[d_op](current_groups, data_dict, k, rng)
        if not removed:
            continue
        candidate_groups = repair_fn[r_op](partial, removed, data_dict, rng)

        # Sanity check: all clinics must be present after repair
        expected = set(range(1, data_dict['num_nodes']))
        if set(j for g in candidate_groups for j in g.all_jobs()) != expected:
            continue

        cand_obj = evaluate_solution(candidate_groups, data_dict)
        if np.isinf(cand_obj):
            continue

        # ── Simulated Annealing acceptance criterion ──────────────────────────
        delta    = cand_obj - current_obj
        accepted = (delta < 0) or (
            temperature > 1e-10 and rng.random() < math.exp(-delta / temperature)
        )

        # ── Determine reward tier ─────────────────────────────────────────────
        if cand_obj < best_obj:
            reward_tier = 0          # new global best
            destroy_new_best[d_op] += 1
            repair_new_best[r_op] += 1
            best_groups = [g.copy() for g in candidate_groups]
            best_obj    = cand_obj
        elif accepted and delta < 0:
            reward_tier = 1          # improved current but not global best
            destroy_improve_current[d_op] += 1
            repair_improve_current[r_op] += 1
        else:
            reward_tier = 2          # accepted worse or rejected

        if accepted:
            current_groups = candidate_groups
            current_obj    = cand_obj

        d_weights.update(d_op, reward_tier)
        r_weights.update(r_op, reward_tier)
        temperature *= cooling

        if verbose and it % 200 == 0:
            elapsed = time.time() - t0
            print(f"  Iter {it:5d} | best={best_obj:.4f}  "
                  f"current={current_obj:.4f}  T={temperature:.4f}  "
                  f"time={elapsed:.1f}s")

    wall_time = time.time() - t0
    if verbose:
        print(f"\nALNS done   | Best corrected obj = {best_obj:.4f}  "
              f"({wall_time:.2f}s, {it} iters)")
        print(f"  Destroy weights : { {op: round(d_weights.weights[op],3) for op in destroy_ops} }")
        print(f"  Repair  weights : { {op: round(r_weights.weights[op],3) for op in repair_ops} }")

    flat_routes = [leg for g in best_groups for leg in g.legs if leg]
    return {
        'objective'        : round(best_obj, 4),
        'routes'           : flat_routes,
        'groups'           : best_groups,
        'n_replenishments' : sum(g.n_replenishments() for g in best_groups),
        'wall_time'        : round(time.time() - t0, 2),
        'iterations'       : it,
        'destroy_weights'  : {op: round(d_weights.weights[op], 4) for op in destroy_ops},
        'repair_weights'   : {op: round(r_weights.weights[op], 4) for op in repair_ops},

        # Diagnostics used by component analysis; they do not affect search.
        'destroy_usage'            : destroy_usage,
        'repair_usage'             : repair_usage,
        'destroy_new_best'         : destroy_new_best,
        'repair_new_best'          : repair_new_best,
        'destroy_improve_current'  : destroy_improve_current,
        'repair_improve_current'   : repair_improve_current,
        'mip_details'              : mip_details,
    }

## 5. Entry Point

Configure and run the solvers.  Three modes are supported:

| MODE    | Description |
|---------|-------------|
| `'mip'` | Run Gurobi MIP on **all** .txt files in `MIP_FOLDER` |
| `'alns'`| Run ALNS (+ optional MIP local search) on `TARGET_FILES` |
| `'both'`| Run both MIP and ALNS on `TARGET_FILES` and compare results |
| `'debug'`| Quick sanity check on a single file (prints distance matrix) |

**Before running**, update `BASE` to point to your local benchmark folder.


In [19]:
# ── Run mode ──────────────────────────────────────────────────────────────────
MODE = 'debug'   # ← change here

# ── Paths ─────────────────────────────────────────────────────────────────────
BASE        = Path('/Users/jane/Desktop/2025-2026-2/Master Thesis')
DEBUG_FOLDER = BASE / 'MSCDPinstances'
MIP_FOLDER  = BASE / 'MSCDPinstances' / '025'
ALNS_FOLDER = BASE / 'MSCDPinstances' / '100'

TARGET_FILES = [
    DEBUG_FOLDER/ '025' / '025_C101.txt', DEBUG_FOLDER/ '025' / '025_C201.txt',
    DEBUG_FOLDER/ '025' / '025_R101.txt', DEBUG_FOLDER/ '025' / '025_R201.txt',
    DEBUG_FOLDER/ '025' / '025_RC101.txt', DEBUG_FOLDER/ '025' / '025_RC201.txt',
    DEBUG_FOLDER / '050' / '050_C101.txt', DEBUG_FOLDER / '050' / '050_C201.txt',
    DEBUG_FOLDER / '050' / '050_R101.txt', DEBUG_FOLDER / '050' / '050_R201.txt',
    DEBUG_FOLDER / '050' / '050_RC101.txt', DEBUG_FOLDER / '050' / '050_RC201.txt',
    DEBUG_FOLDER / '100' / '100_C101.txt', DEBUG_FOLDER / '100' / '100_C201.txt',
    DEBUG_FOLDER / '100' / '100_R101.txt', DEBUG_FOLDER / '100' / '100_R201.txt',
    DEBUG_FOLDER / '100' / '100_RC101.txt', DEBUG_FOLDER / '100' / '100_RC201.txt',
]

MIP_TIME_LIMIT    = 3600
ALNS_TIME_BUDGETS = {'025': 60, '050': 120, '100': 300}

ALNS_PARAMS = dict(
    max_iter=5000,
    removal_min=2,
    cooling=0.9975,
    segment_size=100,
    seed=42,
    verbose=False,

    # Final request-based restricted-MIP setting selected in Section 6.
    mip_local_freq=500,
    mip_local_time=20,
    mip_local_q=8,
    mip_local_min_groups=3,
    mip_local_max_groups=5,
)

# ── Helpers ───────────────────────────────────────────────────────────────────
def _load(fp):
    return preprocess_instance(load_instance(fp))

def _run_mip(fp, data):
    return mip_build_and_solve(data, time_limit=MIP_TIME_LIMIT,
                               threads=4, verbose=False, use_warm_start=False)

def _run_alns(fp, data):
    size_key    = fp.parent.name
    time_limit  = ALNS_TIME_BUDGETS.get(size_key, 120)
    removal_max = max(3, data['num_nodes'] // 8)
    return alns_build_and_solve(data, time_limit=time_limit,
                                removal_max=removal_max, **ALNS_PARAMS)

def _run_mip_folder(folder):
    folder  = Path(folder)
    files   = sorted(f for f in folder.iterdir() if f.suffix == '.txt')
    results, n_optimal = [], 0
    for fp in files:
        print(f'\n--- {fp.name} ---')
        row = {'file': fp.name, 'size': folder.name}
        try:
            res = mip_build_and_solve(_load(fp), time_limit=MIP_TIME_LIMIT,
                                      threads=4, verbose=False, use_warm_start=False)
            row.update({'mip_obj': res['objective'], 'mip_lb': res['lower_bound'],
                        'mip_gap_pct': res['mip_gap_pct'], 'mip_time': res['wall_time'],
                        'mip_optimal': res['proven_optimal']})
            if res['proven_optimal']:
                n_optimal += 1
            gap_str = f"{res['mip_gap_pct']:.1f}%" if res['mip_gap_pct'] is not None else 'N/A'
            print(f"  MIP  Status={res['status']:<10}  Obj={res['objective']}  "
                  f"LB={res['lower_bound']}  Gap={gap_str}  Time={res['wall_time']}s  "
                  f"Optimal={'YES' if res['proven_optimal'] else 'no'}")
        except Exception:
            import traceback; traceback.print_exc()
            row['error'] = True
        results.append(row)
    print(f'\n=== MIP Summary ({folder.name}): {n_optimal}/{len(files)} proven optimal ===')
    return results

# ── Debug ─────────────────────────────────────────────────────────────────────
if MODE == 'debug':
    fp   = TARGET_FILES[0]
    data = _load(fp)
    print(f"Debug: {fp.name}  nodes={data['num_nodes']}  vehicles={data['max_vehicles']}")
    print(data['dist_matrix'])

# ── MIP: all files in MIP_FOLDER ─────────────────────────────────────────────
elif MODE == 'mip':
    print("\n" + "="*60)
    print(f"Running MIP on ALL files in: {MIP_FOLDER}")
    print("="*60)
    all_results = _run_mip_folder(MIP_FOLDER)

    csv_keys = ['file', 'size', 'mip_obj', 'mip_lb', 'mip_gap_pct', 'mip_time', 'mip_optimal']
    print('\n' + '='*80)
    print(f"{'File':<25} {'MIP Obj':>10} {'LB':>10} {'Gap%':>7} {'Time':>8} {'Opt?':>5}")
    print('-'*80)
    for r in all_results:
        print(f"{r['file']:<25} {str(r.get('mip_obj','N/A')):>10} "
              f"{str(r.get('mip_lb','N/A')):>10} {str(r.get('mip_gap_pct','N/A')):>7} "
              f"{str(r.get('mip_time','N/A')):>8} {'YES' if r.get('mip_optimal') else 'no':>5}")
    print('='*80)

# ── ALNS: all files in ALNS_FOLDER ───────────────────────────────────────────
elif MODE == 'alns':
    files = sorted(f for f in ALNS_FOLDER.iterdir() if f.suffix == '.txt')
    print(f"Running ALNS on ALL {len(files)} files in: {ALNS_FOLDER}")
    all_results, total_time, n_error = [], 0.0, 0

    for fp in files:
        print(f'\n--- {fp.name} ---')
        row = {'file': fp.name, 'size': fp.parent.name}
        try:
            res = _run_alns(fp, _load(fp))
            row.update({'alns_obj': res['objective'], 'alns_time': res['wall_time'],
                        'alns_iters': res['iterations'],
                        'alns_dweights': str(res['destroy_weights']),
                        'alns_rweights': str(res['repair_weights'])})
            print(f"  ALNS Obj={res['objective']}  Iters={res['iterations']}  Time={res['wall_time']}s")
            print(f"       Destroy={res['destroy_weights']}")
            print(f"       Repair ={res['repair_weights']}")
        except Exception:
            import traceback; traceback.print_exc()
            n_error += 1; row['error'] = True
        all_results.append(row)
        total_time += row.get('alns_time') or 0.0

    csv_keys = ['file', 'size', 'alns_obj', 'alns_time', 'alns_iters',
                'alns_dweights', 'alns_rweights']
    print('\n' + '='*80)
    print(f"{'File':<25} {'Size':>5} {'ALNS Obj':>10} {'Iters':>7} {'Time(s)':>8}")
    print('-'*80)
    for r in all_results:
        print(f"{r['file']:<25} {r['size']:>5} {str(r.get('alns_obj','N/A')):>10} "
              f"{str(r.get('alns_iters','N/A')):>7} {str(r.get('alns_time','N/A')):>8}")
    print('='*80)
    print(f"Total: {len(all_results)}  |  Errors: {n_error}  |  Time: {total_time:.1f}s ({total_time/60:.1f} min)")

# ── Both: MIP + ALNS on all files in ALNS_FOLDER ─────────────────────────────
elif MODE == 'both':
    files = sorted(f for f in ALNS_FOLDER.iterdir() if f.suffix == '.txt')
    print(f"Running MIP + ALNS on ALL {len(files)} files in: {ALNS_FOLDER}")
    all_results, total_time, n_error = [], 0.0, 0

    for fp in files:
        print(f'\n--- {fp.name} ---')
        row = {'file': fp.name, 'size': fp.parent.name}
        try:
            data = _load(fp)

            mip_res = mip_build_and_solve(data, time_limit=300,
                               threads=4, verbose=False, use_warm_start=False)
            row.update({'mip_obj': mip_res['objective'], 'mip_lb': mip_res['lower_bound'],
                        'mip_gap_pct': mip_res['mip_gap_pct'], 'mip_time': mip_res['wall_time'],
                        'mip_optimal': mip_res['proven_optimal']})
            gap_str = f"{mip_res['mip_gap_pct']:.1f}%" if mip_res['mip_gap_pct'] is not None else 'N/A'
            print(f"  MIP  Status={mip_res['status']:<10}  Obj={mip_res['objective']}  "
                  f"LB={mip_res['lower_bound']}  Gap={gap_str}  Time={mip_res['wall_time']}s  "
                  f"Optimal={'YES' if mip_res['proven_optimal'] else 'no'}")

            alns_res = _run_alns(fp, data)
            row.update({'alns_obj': alns_res['objective'], 'alns_time': alns_res['wall_time'],
                        'alns_iters': alns_res['iterations'],
                        'alns_dweights': str(alns_res['destroy_weights']),
                        'alns_rweights': str(alns_res['repair_weights'])})
            print(f"  ALNS Obj={alns_res['objective']}  Iters={alns_res['iterations']}  Time={alns_res['wall_time']}s")
            print(f"       Destroy={alns_res['destroy_weights']}")
            print(f"       Repair ={alns_res['repair_weights']}")

            if mip_res['lower_bound'] is not None:
                gap_vs_lb = (alns_res['objective'] - mip_res['lower_bound']) / mip_res['lower_bound'] * 100
                row['alns_gap_vs_lb'] = round(gap_vs_lb, 2)
                row['alns_proven_optimal'] = abs(gap_vs_lb) < 0.01
                print(f"       vs LB:  {gap_vs_lb:+.2f}%")

        except Exception:
            import traceback; traceback.print_exc()
            n_error += 1; row['error'] = True
        all_results.append(row)
        total_time += (row.get('alns_time') or 0.0) + (row.get('mip_time') or 0.0)

    csv_keys = ['file', 'size',
                'mip_obj', 'mip_lb', 'mip_gap_pct', 'mip_time', 'mip_optimal',
                'alns_obj', 'alns_time', 'alns_iters', 'alns_gap_vs_lb', 'alns_proven_optimal',
                'alns_dweights', 'alns_rweights']
    print('\n' + '='*80)
    print(f"{'File':<25} {'ALNS':>10} {'MIP':>10} {'vs LB%':>8} {'Opt?':>5}")
    print('-'*80)
    for r in all_results:
        print(f"{r['file']:<25} {str(r.get('alns_obj','N/A')):>10} "
              f"{str(r.get('mip_obj','N/A')):>10} {str(r.get('alns_gap_vs_lb','N/A')):>8} "
              f"{str(r.get('alns_proven_optimal','N/A')):>5} "
              f"{'YES' if r.get('mip_optimal') else 'no':>5}")
    print('='*80)
    print(f"Total: {len(all_results)}  |  Errors: {n_error}  |  Time: {total_time:.1f}s ({total_time/60:.1f} min)")

# ── Save CSV ──────────────────────────────────────────────────────────────────
if MODE != 'debug':
    csv_path = BASE / f'results_{MODE}.csv'
    with open(csv_path, 'w', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=csv_keys, extrasaction='ignore')
        writer.writeheader()
        writer.writerows(all_results)
    print(f'Results saved to {csv_path}')


Debug: 025_C101.txt  nodes=26  vehicles=10
[[        inf 18.68154169 20.61552813 16.1245155  18.11077028 15.13274595
  19.         16.         18.11077028 20.09975124 16.76305461 19.6468827
  38.07886553 30.8058436  39.35733731 36.05551275 40.31128874 33.30165161
  35.35533906 39.05124838 10.         10.19803903 12.16552506 13.
  15.         15.13274595]
 [18.68154169         inf         inf         inf         inf         inf
          inf         inf         inf         inf         inf         inf
          inf         inf         inf         inf         inf         inf
          inf         inf         inf 21.9317122          inf         inf
          inf         inf]
 [20.61552813  2.                 inf         inf         inf         inf
          inf         inf         inf         inf         inf         inf
          inf         inf         inf         inf         inf         inf
          inf         inf         inf 23.43074903 24.75883681         inf
          inf         in

In [26]:
# ============================================================
# Sensitivity Analysis: ALNS Search Effort
# ============================================================

import time
import pandas as pd
from pathlib import Path

SENS_ITERATIONS = [5000, 20000, 50000]
SENS_SEED = 42

SENS_FILES = [
    BASE / 'MSCDPinstances' / '025' / '025_C101.txt',
    BASE / 'MSCDPinstances' / '050' / '050_C101.txt',
    BASE / 'MSCDPinstances' / '100' / '100_C101.txt',
]

SENS_FILES = [f for f in SENS_FILES if f.exists()]

sensitivity_results = []

for fpath in SENS_FILES:

    fname = fpath.name

    print("\n" + "=" * 70)
    print(f"Search-effort sensitivity: {fname}")
    print("=" * 70)

    for max_it in SENS_ITERATIONS:

        # Fresh instance for every run
        data_sens = load_instance(str(fpath))
        data_sens = preprocess_instance(data_sens)

        print(f"\nIteration budget = {max_it:,}")

        start = time.time()

        res = alns_build_and_solve(
            data_sens,

            max_iter=max_it,

            # Effectively disable wall-clock termination.
            # Iteration count is the controlled experimental variable.
            time_limit=100000,

            removal_min=2,
            removal_max=6,
            cooling=0.9975,
            segment_size=100,
            seed=SENS_SEED,
            verbose=False,

            # Pure ALNS sensitivity:
            # exact intensification disabled
            mip_local_freq=0,

            candidate_list_size=20
        )

        runtime = time.time() - start

        sensitivity_results.append({
            'file': fname,
            'iteration_budget': max_it,
            'objective': float(res['objective']),
            'runtime_s': round(runtime, 2),
            'iterations_completed': int(res['iterations']),
            'n_replenishments': int(res['n_replenishments'])
        })

        print(
            f"Objective = {res['objective']:.2f} | "
            f"Runtime = {runtime:.1f}s | "
            f"Iterations = {res['iterations']}"
        )


# ============================================================
# Results
# ============================================================

sens_df = pd.DataFrame(sensitivity_results)

print("\nRAW RESULTS")
display(sens_df)


# ============================================================
# Improvement relative to 5,000 iterations
# ============================================================

comparison = []

for fname in sens_df['file'].unique():

    tmp = (
        sens_df[sens_df['file'] == fname]
        .sort_values('iteration_budget')
    )

    base_obj = float(
        tmp[tmp['iteration_budget'] == 5000]
        .iloc[0]['objective']
    )

    for _, row in tmp.iterrows():

        obj = float(row['objective'])

        improvement = (
            (base_obj - obj) / base_obj * 100
        )

        comparison.append({
            'file': fname,
            'iteration_budget': int(row['iteration_budget']),
            'objective': obj,
            'improvement_vs_5000_%': round(improvement, 2),
            'runtime_s': float(row['runtime_s'])
        })

sens_compare = pd.DataFrame(comparison)

print("\nSOLUTION QUALITY VS. 5,000 ITERATIONS")
display(sens_compare)


# ============================================================
# Average summary
# ============================================================

sens_summary = (
    sens_compare
    .groupby('iteration_budget')
    .agg(
        avg_objective=('objective', 'mean'),
        avg_improvement_vs_5000_pct=(
            'improvement_vs_5000_%', 'mean'
        ),
        avg_runtime_s=('runtime_s', 'mean')
    )
    .reset_index()
)

print("\nAVERAGE SEARCH-EFFORT SENSITIVITY")
display(sens_summary)


# ============================================================
# Save
# ============================================================

sens_df.to_csv(
    BASE / 'only_vehicle_ALNS_iteration_sensitivity.csv',
    index=False
)

sens_summary.to_csv(
    BASE / 'only_vehicle_ALNS_iteration_sensitivity_summary.csv',
    index=False
)


Search-effort sensitivity: 025_C101.txt

Iteration budget = 5,000
Objective = 567.04 | Runtime = 9.0s | Iterations = 5000

Iteration budget = 20,000
Objective = 567.04 | Runtime = 36.3s | Iterations = 20000

Iteration budget = 50,000
Objective = 567.04 | Runtime = 87.1s | Iterations = 50000

Search-effort sensitivity: 050_C101.txt

Iteration budget = 5,000
Objective = 1220.58 | Runtime = 16.7s | Iterations = 5000

Iteration budget = 20,000
Objective = 1220.58 | Runtime = 68.7s | Iterations = 20000

Iteration budget = 50,000
Objective = 1220.58 | Runtime = 164.1s | Iterations = 50000

Search-effort sensitivity: 100_C101.txt

Iteration budget = 5,000
Objective = 3355.38 | Runtime = 25.3s | Iterations = 5000

Iteration budget = 20,000
Objective = 3154.18 | Runtime = 106.7s | Iterations = 20000

Iteration budget = 50,000
Objective = 3118.63 | Runtime = 278.7s | Iterations = 50000

RAW RESULTS


,file,iteration_budget,objective,runtime_s,iterations_completed,n_replenishments
0,025_C101.txt,5000,567.0375,8.97,5000,15
1,025_C101.txt,20000,567.0375,36.27,20000,15
2,025_C101.txt,50000,567.0375,87.14,50000,15
3,050_C101.txt,5000,1220.5824,16.65,5000,35
4,050_C101.txt,20000,1220.5824,68.69,20000,35
5,050_C101.txt,50000,1220.5824,164.09,50000,35
6,100_C101.txt,5000,3355.3817,25.28,5000,62
7,100_C101.txt,20000,3154.1827,106.72,20000,67
8,100_C101.txt,50000,3118.6264,278.75,50000,69



SOLUTION QUALITY VS. 5,000 ITERATIONS


,file,iteration_budget,objective,improvement_vs_5000_%,runtime_s
0,025_C101.txt,5000,567.0375,0.00,8.97
1,025_C101.txt,20000,567.0375,0.00,36.27
2,025_C101.txt,50000,567.0375,0.00,87.14
3,050_C101.txt,5000,1220.5824,0.00,16.65
4,050_C101.txt,20000,1220.5824,0.00,68.69
5,050_C101.txt,50000,1220.5824,0.00,164.09
6,100_C101.txt,5000,3355.3817,0.00,25.28
7,100_C101.txt,20000,3154.1827,6.00,106.72
8,100_C101.txt,50000,3118.6264,7.06,278.75



AVERAGE SEARCH-EFFORT SENSITIVITY


,iteration_budget,avg_objective,avg_improvement_vs_5000_pct,avg_runtime_s
0,5000,1714.333867,0.000000,16.966667
1,20000,1647.267533,2.000000,70.560000
2,50000,1635.415433,2.353333,176.660000


## 6. Component Analysis — Pure ALNS vs. Hybrid ALNS

This experiment isolates the contribution of the **request-based restricted-MIP intensification**.

The comparison is paired:

- **Pure ALNS:** `mip_local_freq = 0`
- **Hybrid ALNS:** `mip_local_freq = 500`, `q = 8`, 3–5 affected groups, 20 seconds per restricted-MIP call
- the same instance, random seed, initialization, destroy/repair operators, adaptive weights, simulated-annealing parameters, candidate-list size, and **ALNS iteration budget** are used in each pair.

The wall-clock limit is set very high so that both variants complete the same number of ALNS iterations. This avoids confounding the component effect with a different number of ALNS iterations. Runtime is reported separately as the computational cost of hybridization.

The main measure is:

`Hybrid improvement (%) = (F'_pure - F'_hybrid) / F'_pure * 100`

A positive value means that exact-assisted intensification improves the final objective. The experiment also reports runtime overhead, number of restricted-MIP calls, and the number of calls that directly improve the incumbent.

The default test set uses representative C101, R101 and RC101 instances at 50 and 100 requests. Three matched seeds are supplied for the final comparison. You can reduce `COMP_SEEDS` to `[42]` for a quick preliminary run.

**This section is manual by default.** Set `RUN_COMPONENT_ANALYSIS = True` in the next cell to execute it.


In [21]:
# =============================================================================
# SECTION 6 — COMPONENT ANALYSIS: PURE ALNS VS HYBRID ALNS
# =============================================================================

import copy
import numpy as np
import pandas as pd
from pathlib import Path

COMP_FILES = [
    BASE / 'MSCDPinstances' / '050' / '050_C101.txt',
    BASE / 'MSCDPinstances' / '050' / '050_R101.txt',
    BASE / 'MSCDPinstances' / '050' / '050_RC101.txt',
    BASE / 'MSCDPinstances' / '100' / '100_C101.txt',
    BASE / 'MSCDPinstances' / '100' / '100_R101.txt',
    BASE / 'MSCDPinstances' / '100' / '100_RC101.txt',
]

# For a quick preliminary run, use [42].
COMP_SEEDS = [42, 43, 44]

COMP_MAX_ITER = 5000
COMP_OUTER_TIME_LIMIT = 100000  # effectively iteration-controlled
COMP_CANDIDATE_LIST_SIZE = 20

# Final hybrid setting
COMP_MIP_FREQ = 500
COMP_MIP_TIME = 20
COMP_MIP_Q = 8
COMP_MIP_MIN_GROUPS = 3
COMP_MIP_MAX_GROUPS = 5


def run_hybrid_component_analysis(
    test_files,
    seeds=(42, 43, 44),
    max_iter=5000,
    outer_time_limit=100000,
    candidate_list_size=20,
    mip_freq=500,
    mip_time=20,
    mip_q=8,
    mip_min_groups=3,
    mip_max_groups=5,
    save_csv=True,
):
    """
    Paired pure-ALNS versus hybrid-ALNS comparison.

    The only algorithmic difference between each pair is whether
    restricted-MIP intensification is enabled.
    """
    rows = []

    existing_files = [Path(fp) for fp in test_files if Path(fp).exists()]
    missing_files = [Path(fp) for fp in test_files if not Path(fp).exists()]

    if missing_files:
        print("Skipped missing files:")
        for fp in missing_files:
            print(f"  - {fp}")

    if not existing_files:
        raise FileNotFoundError(
            "None of the component-analysis files exist. Check BASE and COMP_FILES."
        )

    for fp in existing_files:
        print("\n" + "=" * 100)
        print(f"COMPONENT ANALYSIS: {fp.name}")
        print("=" * 100)

        base_data = _load(fp)
        removal_max = max(3, base_data['num_nodes'] // 8)

        for seed in seeds:
            print(f"\nSeed {seed}")

            pure = alns_build_and_solve(
                copy.deepcopy(base_data),
                max_iter=max_iter,
                time_limit=outer_time_limit,
                removal_min=2,
                removal_max=removal_max,
                cooling=0.9975,
                segment_size=100,
                seed=seed,
                verbose=False,
                mip_local_freq=0,
                candidate_list_size=candidate_list_size,
            )

            hybrid = alns_build_and_solve(
                copy.deepcopy(base_data),
                max_iter=max_iter,
                time_limit=outer_time_limit,
                removal_min=2,
                removal_max=removal_max,
                cooling=0.9975,
                segment_size=100,
                seed=seed,
                verbose=False,
                mip_local_freq=mip_freq,
                mip_local_time=mip_time,
                mip_local_q=mip_q,
                mip_local_min_groups=mip_min_groups,
                mip_local_max_groups=mip_max_groups,
                candidate_list_size=candidate_list_size,
            )

            pure_obj = float(pure['objective'])
            hybrid_obj = float(hybrid['objective'])

            improvement_pct = (
                (pure_obj - hybrid_obj) / pure_obj * 100
                if pure_obj != 0 else np.nan
            )

            pure_time = float(pure['wall_time'])
            hybrid_time = float(hybrid['wall_time'])
            runtime_overhead_pct = (
                (hybrid_time - pure_time) / pure_time * 100
                if pure_time > 0 else np.nan
            )

            mip_details = hybrid.get('mip_details', [])
            mip_calls = len(mip_details)
            mip_improvements = sum(
                1 for d in mip_details if d.get('improved', False)
            )
            mip_seconds = sum(
                float(d.get('mip_time', 0.0) or 0.0) for d in mip_details
            )

            rows.append({
                'file': fp.name,
                'size': fp.parent.name,
                'seed': int(seed),
                'pure_obj': pure_obj,
                'hybrid_obj': hybrid_obj,
                'hybrid_improvement_pct': improvement_pct,
                'hybrid_better': hybrid_obj < pure_obj - 1e-6,
                'same_objective': abs(hybrid_obj - pure_obj) <= 1e-6,
                'pure_runtime_s': pure_time,
                'hybrid_runtime_s': hybrid_time,
                'runtime_overhead_pct': runtime_overhead_pct,
                'pure_iterations': int(pure['iterations']),
                'hybrid_iterations': int(hybrid['iterations']),
                'mip_calls': mip_calls,
                'mip_improvements': mip_improvements,
                'mip_improvement_rate_pct':
                    (100.0 * mip_improvements / mip_calls) if mip_calls else 0.0,
                'restricted_mip_time_s': mip_seconds,
            })

            print(
                f"  Pure   F'={pure_obj:.2f} | {pure_time:.1f}s | "
                f"{pure['iterations']} iters"
            )
            print(
                f"  Hybrid F'={hybrid_obj:.2f} | {hybrid_time:.1f}s | "
                f"{hybrid['iterations']} iters"
            )
            print(
                f"  Hybrid improvement={improvement_pct:+.2f}% | "
                f"runtime overhead={runtime_overhead_pct:+.1f}% | "
                f"MIP improved {mip_improvements}/{mip_calls} calls"
            )

    detail_df = pd.DataFrame(rows)

    summary_df = (
        detail_df
        .groupby(['size', 'file'], as_index=False)
        .agg(
            seeds=('seed', 'nunique'),
            mean_pure_obj=('pure_obj', 'mean'),
            mean_hybrid_obj=('hybrid_obj', 'mean'),
            mean_hybrid_improvement_pct=('hybrid_improvement_pct', 'mean'),
            std_hybrid_improvement_pct=('hybrid_improvement_pct', 'std'),
            hybrid_win_rate_pct=('hybrid_better', lambda s: 100.0 * s.mean()),
            mean_pure_runtime_s=('pure_runtime_s', 'mean'),
            mean_hybrid_runtime_s=('hybrid_runtime_s', 'mean'),
            mean_runtime_overhead_pct=('runtime_overhead_pct', 'mean'),
            mip_calls=('mip_calls', 'sum'),
            mip_improvements=('mip_improvements', 'sum'),
            restricted_mip_time_s=('restricted_mip_time_s', 'sum'),
        )
    )
    summary_df['mip_improvement_rate_pct'] = np.where(
        summary_df['mip_calls'] > 0,
        100.0 * summary_df['mip_improvements'] / summary_df['mip_calls'],
        0.0
    )

    size_summary_df = (
        detail_df
        .groupby('size', as_index=False)
        .agg(
            runs=('seed', 'size'),
            mean_hybrid_improvement_pct=('hybrid_improvement_pct', 'mean'),
            median_hybrid_improvement_pct=('hybrid_improvement_pct', 'median'),
            hybrid_win_rate_pct=('hybrid_better', lambda s: 100.0 * s.mean()),
            mean_pure_runtime_s=('pure_runtime_s', 'mean'),
            mean_hybrid_runtime_s=('hybrid_runtime_s', 'mean'),
            mean_runtime_overhead_pct=('runtime_overhead_pct', 'mean'),
            mip_calls=('mip_calls', 'sum'),
            mip_improvements=('mip_improvements', 'sum'),
        )
    )
    size_summary_df['mip_improvement_rate_pct'] = np.where(
        size_summary_df['mip_calls'] > 0,
        100.0 * size_summary_df['mip_improvements'] / size_summary_df['mip_calls'],
        0.0
    )

    overall_df = pd.DataFrame([{
        'runs': len(detail_df),
        'instances': detail_df['file'].nunique(),
        'mean_hybrid_improvement_pct':
            detail_df['hybrid_improvement_pct'].mean(),
        'median_hybrid_improvement_pct':
            detail_df['hybrid_improvement_pct'].median(),
        'hybrid_win_rate_pct':
            100.0 * detail_df['hybrid_better'].mean(),
        'same_objective_rate_pct':
            100.0 * detail_df['same_objective'].mean(),
        'mean_runtime_overhead_pct':
            detail_df['runtime_overhead_pct'].mean(),
        'total_mip_calls':
            int(detail_df['mip_calls'].sum()),
        'total_mip_improvements':
            int(detail_df['mip_improvements'].sum()),
        'mip_improvement_rate_pct':
            (
                100.0 * detail_df['mip_improvements'].sum()
                / detail_df['mip_calls'].sum()
                if detail_df['mip_calls'].sum() else 0.0
            ),
    }])

    print("\n" + "=" * 100)
    print("INSTANCE-LEVEL SUMMARY")
    print("=" * 100)
    print(summary_df.round(2).to_string(index=False))

    print("\n" + "=" * 100)
    print("SIZE-LEVEL SUMMARY")
    print("=" * 100)
    print(size_summary_df.round(2).to_string(index=False))

    print("\n" + "=" * 100)
    print("OVERALL COMPONENT EFFECT")
    print("=" * 100)
    print(overall_df.round(2).to_string(index=False))

    if save_csv:
        detail_path = BASE / 'only_vehicle_component_pure_vs_hybrid_detail.csv'
        summary_path = BASE / 'only_vehicle_component_pure_vs_hybrid_summary.csv'
        size_path = BASE / 'only_vehicle_component_pure_vs_hybrid_size_summary.csv'
        overall_path = BASE / 'only_vehicle_component_pure_vs_hybrid_overall.csv'

        detail_df.to_csv(detail_path, index=False)
        summary_df.to_csv(summary_path, index=False)
        size_summary_df.to_csv(size_path, index=False)
        overall_df.to_csv(overall_path, index=False)

        print("\nSaved:")
        print(f"  {detail_path}")
        print(f"  {summary_path}")
        print(f"  {size_path}")
        print(f"  {overall_path}")

    return detail_df, summary_df, size_summary_df, overall_df


# Keep False for normal Run All. Set True only for this experiment.
RUN_COMPONENT_ANALYSIS = True

if RUN_COMPONENT_ANALYSIS:
    component_detail_df, component_summary_df, component_size_df, component_overall_df = (
        run_hybrid_component_analysis(
            COMP_FILES,
            seeds=COMP_SEEDS,
            max_iter=COMP_MAX_ITER,
            outer_time_limit=COMP_OUTER_TIME_LIMIT,
            candidate_list_size=COMP_CANDIDATE_LIST_SIZE,
            mip_freq=COMP_MIP_FREQ,
            mip_time=COMP_MIP_TIME,
            mip_q=COMP_MIP_Q,
            mip_min_groups=COMP_MIP_MIN_GROUPS,
            mip_max_groups=COMP_MIP_MAX_GROUPS,
            save_csv=True,
        )
    )



COMPONENT ANALYSIS: 050_C101.txt

Seed 42
  Pure   F'=1220.58 | 17.7s | 5000 iters
  Hybrid F'=1238.44 | 99.8s | 5000 iters
  Hybrid improvement=-1.46% | runtime overhead=+465.1% | MIP improved 5/10 calls

Seed 43
  Pure   F'=1255.19 | 17.9s | 5000 iters
  Hybrid F'=1239.01 | 69.3s | 5000 iters
  Hybrid improvement=+1.29% | runtime overhead=+286.2% | MIP improved 2/10 calls

Seed 44
  Pure   F'=1207.85 | 18.4s | 5000 iters
  Hybrid F'=1207.85 | 79.7s | 5000 iters
  Hybrid improvement=+0.00% | runtime overhead=+332.9% | MIP improved 4/10 calls

COMPONENT ANALYSIS: 050_R101.txt

Seed 42
  Pure   F'=1536.09 | 15.0s | 5000 iters
  Hybrid F'=1455.76 | 16.6s | 5000 iters
  Hybrid improvement=+5.23% | runtime overhead=+10.8% | MIP improved 7/10 calls

Seed 43
  Pure   F'=1571.35 | 15.1s | 5000 iters
  Hybrid F'=1527.54 | 16.9s | 5000 iters
  Hybrid improvement=+2.79% | runtime overhead=+11.9% | MIP improved 4/10 calls

Seed 44
  Pure   F'=1468.46 | 14.2s | 5000 iters
  Hybrid F'=1463.65 | 16

## 7. Parameter Tuning and Operator Diagnostics — manual only

**This section is diagnostic only. A normal `Run All` only defines the functions below; it does not launch any tuning experiment.**

To run an experiment, manually call one of the helper functions in the last cell and pass the files you want to test.

### 7.1 Development path of the restricted MIP

The exact-intensification design was developed in stages.

**Stage A — route-based neighborhoods**

- Random 1-RouteGroup reoptimization: almost no direct improvements.
- Random 2-RouteGroup reoptimization: 0 improvements in 30 calls on representative C/R/RC instances.
- Targeted 2-RouteGroup reoptimization: again 0 improvements in 30 calls.

These tests indicated that fixing all requests outside one or two complete RouteGroups created a neighborhood that was too restrictive.

**Stage B — request-induced neighborhood**

The neighborhood was changed to select high-cost requests first and then jointly reoptimize the vehicles containing them. With 12 target requests, 3–5 affected vehicles, frequency 500 and a 20-second exact budget, 13 of 30 restricted-MIP calls directly improved the incumbent on C101/R101/RC101.

### 7.2 Parameter tests

Representative tests produced the following observations:

- `q=8, freq=1000, 20s`: average objective change versus pure ALNS **+1.52%**; 11/15 MIP calls improved the incumbent.
- `q=12, freq=1000, 20s`: essentially identical to `q=8`, showing that 8 and 12 requests often induced the same affected vehicles.
- `q=8, freq=500, 20s`: average objective change **-0.45%**; 13/30 calls improved the incumbent. This was the best tested average objective configuration, but gains were instance-dependent and runtime increased.
- Intermediate frequency (about 750): worse final objectives on the three representative instances.
- `q=8, freq=500, 10s`: removed most of the C-instance benefit; all three representative final objectives were worse than pure ALNS.

Therefore the production setting used in Section 5 is:

\[
q=8,\qquad f=500,\qquad t_{\text{MIP}}=20\text{ s},
\]

with 3–5 affected vehicle groups.

This is the **best tested hybrid configuration**, not a claim of universal optimality.

### 7.3 Operator diagnostics

Across three representative 100-request pure-ALNS runs (15,000 operator selections in total):

| Type | Operator | Selection share | New-best rate | Current-improvement rate |
|---|---|---:|---:|---:|
| Destroy | Random | 33.02% | 1.94% | 9.55% |
| Destroy | Worst | 33.37% | 1.60% | 8.59% |
| Destroy | Shaw | 33.61% | 1.65% | 6.33% |
| Repair | Regret-2 | 50.11% | 1.82% | 8.58% |
| Repair | Greedy | 49.89% | 1.63% | 7.71% |

The adaptive mechanism therefore maintained nearly balanced selection frequencies. Random removal produced the strongest overall destroy-side improvement rates, while regret-2 slightly outperformed greedy insertion.

### 7.4 Manual use

The code cell below only defines helper functions. Example calls are intentionally left commented out so that `Run All` remains safe.

You can freely replace the files in `test_files`, for example:

```python
test_files = [
    BASE / 'MSCDPinstances' / '100' / '100_C101.txt',
    BASE / 'MSCDPinstances' / '100' / '100_R101.txt',
    BASE / 'MSCDPinstances' / '100' / '100_RC101.txt',
]

# run_parameter_tuning(test_files)
# run_operator_diagnostics(test_files)
```


In [ ]:
# =============================================================================
# SECTION 7 — MANUAL DIAGNOSTICS ONLY
# =============================================================================

from collections import defaultdict
import copy
import pandas as pd
import numpy as np


def run_parameter_tuning(
    test_files,
    parameter_sets=None,
    max_iter=5000,
    outer_time_limit=600,
    seed=42,
    mip_time=20,
    min_groups=3,
    max_groups=5,
    candidate_list_size=20,
):
    """
    Manually compare request-based restricted-MIP parameter settings.

    Parameters
    ----------
    test_files : iterable of pathlib.Path
        Any instance files you want to test.
    parameter_sets : list of dicts or None
        Each dict must contain:
            {'name': str, 'q_requests': int, 'mip_freq': int}
        If None, the historical comparison set is used.

    Returns
    -------
    pandas.DataFrame
        One row per (instance, parameter setting).
    """
    if parameter_sets is None:
        parameter_sets = [
            {'name': 'q8_f1000',  'q_requests': 8,  'mip_freq': 1000},
            {'name': 'q12_f1000', 'q_requests': 12, 'mip_freq': 1000},
            {'name': 'q8_f500',   'q_requests': 8,  'mip_freq': 500},
        ]

    rows = []

    for fp in test_files:
        print("\n" + "=" * 100)
        print(f"PARAMETER TUNING: {fp.name}")
        print("=" * 100)

        data = _load(fp)
        removal_max = max(3, data['num_nodes'] // 8)

        pure = alns_build_and_solve(
            copy.deepcopy(data),
            max_iter=max_iter,
            time_limit=outer_time_limit,
            removal_min=2,
            removal_max=removal_max,
            cooling=0.9975,
            segment_size=100,
            seed=seed,
            verbose=False,
            mip_local_freq=0,
            candidate_list_size=candidate_list_size,
        )

        for p in parameter_sets:
            hybrid = alns_build_and_solve(
                copy.deepcopy(data),
                max_iter=max_iter,
                time_limit=outer_time_limit,
                removal_min=2,
                removal_max=removal_max,
                cooling=0.9975,
                segment_size=100,
                seed=seed,
                verbose=False,
                mip_local_freq=p['mip_freq'],
                mip_local_time=mip_time,
                mip_local_q=p['q_requests'],
                mip_local_min_groups=min_groups,
                mip_local_max_groups=max_groups,
                candidate_list_size=candidate_list_size,
            )

            delta = hybrid['objective'] - pure['objective']
            delta_pct = (
                delta / pure['objective'] * 100
                if pure['objective'] != 0 else np.nan
            )

            mip_calls = len(hybrid.get('mip_details', []))
            mip_improvements = sum(
                1 for d in hybrid.get('mip_details', [])
                if d.get('improved')
            )
            mip_seconds = sum(
                d.get('mip_time', 0.0)
                for d in hybrid.get('mip_details', [])
            )

            rows.append({
                'file': fp.name,
                'setting': p['name'],
                'q': p['q_requests'],
                'freq': p['mip_freq'],
                'pure_obj': pure['objective'],
                'hybrid_obj': hybrid['objective'],
                'delta_pct': delta_pct,
                'pure_time': pure['wall_time'],
                'hybrid_time': hybrid['wall_time'],
                'mip_calls': mip_calls,
                'mip_improvements': mip_improvements,
                'mip_improvement_rate_pct':
                    (100 * mip_improvements / mip_calls) if mip_calls else 0.0,
                'mip_time': mip_seconds,
            })

    df = pd.DataFrame(rows)
    if not df.empty:
        print("\nPARAMETER SUMMARY")
        print(df.round(2).to_string(index=False))

        avg = (
            df.groupby('setting', as_index=False)
              .agg(
                  avg_delta_pct=('delta_pct', 'mean'),
                  avg_hybrid_time=('hybrid_time', 'mean'),
                  mip_calls=('mip_calls', 'sum'),
                  mip_improvements=('mip_improvements', 'sum'),
                  total_mip_time=('mip_time', 'sum'),
              )
        )
        avg['mip_improvement_rate_pct'] = np.where(
            avg['mip_calls'] > 0,
            100 * avg['mip_improvements'] / avg['mip_calls'],
            0.0
        )
        print("\nAVERAGE BY SETTING")
        print(avg.round(2).to_string(index=False))

    return df


def run_operator_diagnostics(
    test_files,
    max_iter=5000,
    outer_time_limit=600,
    seed=42,
    candidate_list_size=20,
):
    """
    Run pure ALNS on manually selected files and aggregate operator usage.

    Returns
    -------
    detail_df, summary_df : pandas.DataFrame, pandas.DataFrame
    """
    rows = []

    for fp in test_files:
        print("\n" + "=" * 90)
        print(f"OPERATOR DIAGNOSTIC: {fp.name}")
        print("=" * 90)

        data = _load(fp)
        removal_max = max(3, data['num_nodes'] // 8)

        res = alns_build_and_solve(
            copy.deepcopy(data),
            max_iter=max_iter,
            time_limit=outer_time_limit,
            removal_min=2,
            removal_max=removal_max,
            cooling=0.9975,
            segment_size=100,
            seed=seed,
            verbose=False,
            mip_local_freq=0,
            candidate_list_size=candidate_list_size,
        )

        for op_type, usage_key, best_key, improve_key in [
            ('Destroy', 'destroy_usage', 'destroy_new_best', 'destroy_improve_current'),
            ('Repair',  'repair_usage',  'repair_new_best',  'repair_improve_current'),
        ]:
            total_usage = sum(res[usage_key].values())

            for op, usage in res[usage_key].items():
                new_best = res[best_key].get(op, 0)
                improve_current = res[improve_key].get(op, 0)

                rows.append({
                    'instance': fp.name,
                    'type': op_type,
                    'operator': op,
                    'usage': usage,
                    'selection_share_pct':
                        100 * usage / total_usage if total_usage else 0.0,
                    'new_best': new_best,
                    'new_best_rate_pct':
                        100 * new_best / usage if usage else 0.0,
                    'improve_current': improve_current,
                    'improve_current_rate_pct':
                        100 * improve_current / usage if usage else 0.0,
                })

    detail_df = pd.DataFrame(rows)

    summary_rows = []
    if not detail_df.empty:
        for (op_type, op), sub in detail_df.groupby(['type', 'operator']):
            total_usage = int(sub['usage'].sum())
            total_new_best = int(sub['new_best'].sum())
            total_improve = int(sub['improve_current'].sum())

            summary_rows.append({
                'type': op_type,
                'operator': op,
                'total_usage': total_usage,
                'avg_selection_share_pct': sub['selection_share_pct'].mean(),
                'total_new_best': total_new_best,
                'new_best_rate_pct':
                    100 * total_new_best / total_usage if total_usage else 0.0,
                'total_improve_current': total_improve,
                'improve_current_rate_pct':
                    100 * total_improve / total_usage if total_usage else 0.0,
            })

    summary_df = pd.DataFrame(summary_rows)

    if not summary_df.empty:
        order = {'Destroy': 0, 'Repair': 1}
        summary_df['_order'] = summary_df['type'].map(order)
        summary_df = (
            summary_df
            .sort_values(['_order', 'total_usage'], ascending=[True, False])
            .drop(columns='_order')
        )

    print("\nAGGREGATED OPERATOR SUMMARY")
    if not summary_df.empty:
        print(summary_df.round(2).to_string(index=False))

    return detail_df, summary_df


# -------------------------------------------------------------------------
# MANUAL EXAMPLE — intentionally commented out
# -------------------------------------------------------------------------
# test_files = [
#     BASE / 'MSCDPinstances' / '100' / '100_C101.txt',
#     BASE / 'MSCDPinstances' / '100' / '100_R101.txt',
#     BASE / 'MSCDPinstances' / '100' / '100_RC101.txt',
# ]
#
# tuning_df = run_parameter_tuning(test_files)
# operator_detail_df, operator_summary_df = run_operator_diagnostics(test_files)
